# STM32F429 Audio (SAI/I2S) Tutorial for Beginners

This tutorial provides a comprehensive guide to understanding and using the audio peripherals on STM32F429 microcontrollers. We'll cover everything from basic audio concepts to advanced features like DMA streaming and codec integration.

## Table of Contents
1. [Introduction to Audio Systems](#introduction)
2. [STM32F429 Audio Overview](#audio-overview)
3. [Hardware Setup](#hardware-setup)
4. [Software Dependencies](#software-setup)
5. [Basic Audio Playback](#basic-playback)
6. [Audio Recording](#audio-recording)
7. [Volume and Gain Control](#volume-control)
8. [Multi-Channel Audio](#multi-channel)
9. [DMA Audio Streaming](#dma-streaming)
10. [Audio Processing](#audio-processing)
11. [Codec Integration](#codec-integration)
12. [Advanced Features](#advanced)
13. [Troubleshooting](#troubleshooting)

<a id='introduction'></a>
## 1. Introduction to Audio Systems

Audio processing involves converting between analog sound waves and digital representations. The STM32F429 supports professional audio interfaces for high-quality audio applications.

### Key Audio Concepts:
- **Sample Rate**: How many audio samples per second (44.1kHz, 48kHz, 96kHz)
- **Bit Depth**: Resolution of each sample (16-bit, 24-bit, 32-bit)
- **Channels**: Mono (1) or Stereo (2) audio
- **PCM**: Pulse Code Modulation - standard digital audio format
- **SAI**: Serial Audio Interface - advanced audio protocol
- **I2S**: Inter-IC Sound - standard audio protocol

### Audio Data Flow:
```
Analog Audio → ADC → Digital Samples → Processing → DAC → Analog Audio
```

### Sample Rate Standards:
- **8kHz**: Telephone quality
- **16kHz**: Voice recording
- **44.1kHz**: CD quality
- **48kHz**: Professional audio
- **96kHz**: High-resolution audio

### Bit Depth Ranges:
- **16-bit**: -32,768 to +32,767 (CD quality)
- **24-bit**: -8,388,608 to +8,388,607 (professional)
- **32-bit**: -2,147,483,648 to +2,147,483,647 (floating point equivalent)

<a id='audio-overview'></a>
## 2. STM32F429 Audio Overview

The STM32F429 features advanced audio peripherals supporting professional audio applications.

### Audio Interfaces:
- **SAI (Serial Audio Interface)**: Advanced audio protocol with TDM support
- **I2S (Inter-IC Sound)**: Standard audio protocol for consumer devices
- **SPDIF**: Digital audio interface for optical transmission

### Key Specifications:
- **Sample Rates**: 8kHz to 96kHz
- **Bit Depths**: 16-bit, 24-bit, 32-bit
- **Channels**: Mono and Stereo support
- **DMA Support**: Efficient data transfer without CPU intervention
- **Buffer Management**: Circular buffers for continuous streaming
- **Audio Processing**: Built-in mixing, filtering, and gain control

### SAI vs I2S Comparison:

| Feature | SAI | I2S |
|---------|-----|-----|
| **Protocol** | Advanced | Standard |
| **Channels** | Up to 16 (TDM) | 2 (Stereo) |
| **Flexibility** | High | Medium |
| **Compatibility** | Professional | Consumer |
| **Complexity** | Higher | Lower |

### Audio Data Formats:
- **PCM**: Linear pulse code modulation
- **TDM**: Time division multiplexing (SAI)
- **Left/Right Justified**: Alternative formats
- **DSP Mode**: Digital signal processor format

### Audio Buffer Management:
- **Circular Buffers**: Continuous audio streaming
- **Double Buffering**: Seamless playback/recording
- **DMA Transfers**: CPU-free data movement
- **Interrupt Handling**: Buffer status notifications

### Audio Register Map (Reference Manual)

The STM32F429 SAI peripheral uses several registers for configuration:

| Register | Address Offset | Description |
|----------|----------------|-------------|
| SAI_GCR  | 0x00 | Global Configuration Register |
| SAI_xCR1 | 0x04 | Configuration Register 1 |
| SAI_xCR2 | 0x08 | Configuration Register 2 |
| SAI_xFRCR| 0x0C | Frame Configuration Register |
| SAI_xSLOTR| 0x10 | Slot Register |
| SAI_xIMR | 0x14 | Interrupt Mask Register |
| SAI_xSR  | 0x18 | Status Register |
| SAI_xCLRFR| 0x1C | Clear Flag Register |
| SAI_xDR  | 0x20 | Data Register |

### Key SAI Register Configurations:

#### SAI_xCR1 (Configuration Register 1):
- **Bits 8-5**: MODE[3:0] - Audio mode selection
- **Bits 4-2**: PRTCFG[2:0] - Protocol configuration
- **Bits 1-0**: DS[1:0] - Data size (16/24/32-bit)

#### SAI_xCR2 (Configuration Register 2):
- **Bit 0**: FTH[2:0] - FIFO threshold
- **Bit 3**: TRIS - Transmit receive interrupt
- **Bit 4**: MUTE - Mute mode
- **Bit 5**: MUTECNT - Mute counter

#### SAI_xFRCR (Frame Configuration Register):
- **Bits 7-0**: FRL[7:0] - Frame length
- **Bits 15-8**: FSALL[7:0] - Frame sync active level length
- **Bit 16**: FSDEF - Frame sync definition
- **Bits 19-17**: FSPOL - Frame sync polarity

#### SAI_xSLOTR (Slot Register):
- **Bits 4-0**: NBSLOT[4:0] - Number of slots
- **Bits 11-6**: SLOTEN[5:0] - Slot enable
- **Bits 15-12**: SLOTSZ[1:0] - Slot size

<a id='hardware-setup'></a>
## 3. Hardware Setup

### Required Components:
- STM32F429 Discovery board
- Audio codec (WM8994, CS42L51, or similar)
- Audio amplifier (optional, for speaker output)
- Speakers or headphones
- Microphone (for recording)
- Connecting wires and audio jacks

### SAI Interface Pin Configuration:

| SAI Signal | STM32F429 Pin | Function | Description |
|------------|----------------|----------|-------------|
| SAI1_MCK  | PE2 | Master Clock | Audio master clock output |
| SAI1_SD   | PE4 | Serial Data | Audio data line (bidirectional) |
| SAI1_FS   | PE5 | Frame Sync | Left/Right clock |
| SAI1_SCK  | PE6 | Serial Clock | Audio bit clock |

### I2S Interface Pin Configuration (Alternative):

| I2S Signal | STM32F429 Pin | Function | Description |
|------------|----------------|----------|-------------|
| I2S3_WS   | PA4 | Word Select | Left/Right clock |
| I2S3_CK   | PC10 | Serial Clock | Audio bit clock |
| I2S3_SD   | PC12 | Serial Data | Audio data output |
| I2S3_MCK  | PC7 | Master Clock | Audio master clock |

### Codec Control Interface:
- **I2C1_SCL**: PB6 (Codec control clock)
- **I2C1_SDA**: PB7 (Codec control data)

### Hardware Considerations:
- **Power Supply**: Clean, stable power for audio quality
- **Ground Planes**: Separate analog and digital grounds
- **Clock Source**: External oscillator for accurate audio timing
- **EMI Shielding**: Shield audio signals from digital noise
- **Anti-aliasing**: Proper filtering for ADC/DAC

### Audio Codec Integration:
```
STM32F429 SAI/I2S ──── Audio Codec ──── Amplifier ──── Speakers
       │                       │              │
       └── I2C Control ────────┘              └── Headphones
```

### Circuit Examples:

#### Basic Audio Output Circuit:
```
STM32 SAI_SD ──── Codec DIN
STM32 SAI_SCK ──── Codec BCLK
STM32 SAI_FS ──── Codec LRCLK
STM32 SAI_MCK ──── Codec MCLK
STM32 I2C_SCL ──── Codec SCL
STM32 I2C_SDA ──── Codec SDA

Codec LOUT ──── Amplifier Input ──── Speaker
Codec ROUT ──── Amplifier Input ──── Speaker
```

#### Audio Input Circuit:
```
Microphone ──── Preamp ──── Codec LIN
                        └─── Codec RIN
```

### Clock Configuration:
- **Audio PLL**: Configure PLL for accurate sample rates
- **MCK Output**: Master clock for codec synchronization
- **Sample Rate**: Match codec and STM32 configurations
- **Bit Clock**: 64x sample rate for 32-bit, 32x for 16-bit

### Power Management:
- **Audio Power**: Dedicated power supply for audio circuitry
- **Codec Power**: Proper sequencing for codec initialization
- **Amplifier Power**: Enable/disable for power saving

<a id='software-setup'></a>
## 4. Software Dependencies

### Required Files:
- `audio.h` - Header file with function prototypes and types
- `audio.c` - Main audio driver implementation
- STM32F4xx HAL Library (`stm32f4xx_hal.h`, `stm32f4xx_hal_sai.h`, `stm32f4xx_hal_i2s.h`)
- Standard C libraries: `stdint.h`, `stdbool.h`, `string.h`, `stdlib.h`

### Include Path Setup:
```c
#include "audio.h"
#include "stm32f4xx.h"  // HAL library
```

### Build Configuration:
- Add source files to project: `audio.c`
- Include header paths: path to `audio.h`
- Enable SAI/I2S peripheral in STM32CubeMX
- Configure audio PLL for proper clocking
- Enable DMA for audio streaming
- Configure I2C for codec control

### Audio Handle Declaration:
```c
// Global audio handle for SAI interface
AUDIO_HandleTypeDef haudio_sai;

// Global audio handle for I2S interface
AUDIO_HandleTypeDef haudio_i2s;
```

### Error Handling:
- All functions return `AUDIO_StatusTypeDef`
- Check return values: `AUDIO_OK`, `AUDIO_ERROR`, `AUDIO_BUSY`, `AUDIO_TIMEOUT`
- Use `AUDIO_GetStatusString()` for readable error messages

<a id='basic-playback'></a>
## 5. Basic Audio Playback

### Audio Configuration Structure:
```c
AUDIO_ConfigTypeDef audio_config = {
    .interface = AUDIO_INTERFACE_SAI,        // SAI or I2S
    .sample_rate = AUDIO_SAMPLE_RATE_44_1K,  // 44.1kHz
    .bit_depth = AUDIO_BIT_DEPTH_16,         // 16-bit
    .channels = AUDIO_CHANNELS_STEREO,       // Stereo
    .volume = 80,                            // 80% volume
    .dma_enabled = true                      // Use DMA
};
```

### Basic Playback Initialization:
```c
#include "audio.h"

int main(void) {
    // Configure audio for playback
    AUDIO_ConfigTypeDef config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_48K,
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_STEREO,
        .volume = 70,
        .dma_enabled = false  // Polling mode for simplicity
    };
    
    // Initialize audio system
    AUDIO_StatusTypeDef status = AUDIO_Init(&haudio_sai, &config);
    if (status != AUDIO_OK) {
        printf("Audio initialization failed: %s\n", 
               AUDIO_GetStatusString(status));
        return -1;
    }
    
    printf("Audio system initialized successfully\n");
    
    // Main application loop
    while (1) {
        // Application code here
    }
}
```

### Simple Tone Generation:
```c
// Generate a simple sine wave tone
#define SAMPLE_RATE 48000
#define FREQUENCY 440  // A4 note
#define AMPLITUDE 16384  // For 16-bit audio

void generate_tone(int16_t* buffer, uint32_t samples) {
    static float phase = 0.0f;
    float phase_increment = 2.0f * 3.14159f * FREQUENCY / SAMPLE_RATE;
    
    for (uint32_t i = 0; i < samples; i++) {
        buffer[i] = (int16_t)(AMPLITUDE * sinf(phase));
        phase += phase_increment;
        if (phase >= 2.0f * 3.14159f) {
            phase -= 2.0f * 3.14159f;
        }
    }
}

// Play the generated tone
int16_t audio_buffer[1024];  // 1KB buffer for stereo 16-bit

generate_tone(audio_buffer, 512);  // Generate 512 samples (mono)

AUDIO_StatusTypeDef status = AUDIO_PlayBuffer(&haudio_sai, 
                                             (uint8_t*)audio_buffer, 
                                             1024);  // 1024 bytes for stereo
```

### Playback Control:
```c
// Start playback
AUDIO_StartPlayback(&haudio_sai);

// Pause playback
AUDIO_PausePlayback(&haudio_sai);

// Resume playback
AUDIO_ResumePlayback(&haudio_sai);

// Stop playback
AUDIO_StopPlayback(&haudio_sai);
```

### Understanding Audio Data:
- **16-bit Stereo**: Each sample is 4 bytes (Left + Right channels)
- **Sample Rate**: 48kHz = 48,000 samples per second per channel
- **Data Rate**: 48kHz × 4 bytes = 192kB/s
- **Buffer Size**: Should be multiple of frame size (channels × bit_depth/8)

<a id='audio-recording'></a>
## 6. Audio Recording

### Recording Configuration:
```c
AUDIO_ConfigTypeDef record_config = {
    .interface = AUDIO_INTERFACE_SAI,
    .sample_rate = AUDIO_SAMPLE_RATE_16K,   // 16kHz for voice
    .bit_depth = AUDIO_BIT_DEPTH_16,
    .channels = AUDIO_CHANNELS_MONO,        // Mono for simplicity
    .volume = 100,                          // Maximum input gain
    .dma_enabled = true                     // DMA for continuous recording
};
```

### Basic Recording Setup:
```c
#include "audio.h"

#define RECORD_BUFFER_SIZE 4096
uint8_t record_buffer[RECORD_BUFFER_SIZE];

int main(void) {
    // Configure for recording
    AUDIO_ConfigTypeDef config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_16K,
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_MONO,
        .volume = 100,
        .dma_enabled = false
    };
    
    AUDIO_Init(&haudio_sai, &config);
    
    // Start recording
    AUDIO_StatusTypeDef status = AUDIO_StartRecording(&haudio_sai, 
                                                     record_buffer, 
                                                     RECORD_BUFFER_SIZE);
    if (status == AUDIO_OK) {
        printf("Recording started...\n");
        
        // Record for 2 seconds
        HAL_Delay(2000);
        
        // Stop recording
        AUDIO_StopRecording(&haudio_sai);
        printf("Recording stopped\n");
        
        // Process recorded data
        process_audio_data(record_buffer, RECORD_BUFFER_SIZE);
    }
}
```

### Recording with DMA (Continuous):
```c
// Circular buffer for continuous recording
#define CIRCULAR_BUFFER_SIZE 8192
uint8_t circular_buffer[CIRCULAR_BUFFER_SIZE];
volatile uint32_t record_index = 0;

void start_continuous_recording(void) {
    AUDIO_ConfigTypeDef config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_48K,
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_STEREO,
        .dma_enabled = true,
        .buffer_mode = AUDIO_BUFFER_CIRCULAR
    };
    
    AUDIO_Init(&haudio_sai, &config);
    AUDIO_StartRecording(&haudio_sai, circular_buffer, CIRCULAR_BUFFER_SIZE);
}

void process_recorded_data(void) {
    // Process latest recorded samples
    static uint32_t last_index = 0;
    
    while (last_index != record_index) {
        // Process audio sample
        int16_t sample = (int16_t)(circular_buffer[last_index] | 
                                  (circular_buffer[last_index + 1] << 8));
        
        // Apply processing (echo, filter, etc.)
        sample = process_sample(sample);
        
        last_index = (last_index + 2) % CIRCULAR_BUFFER_SIZE;  // 16-bit = 2 bytes
    }
}
```

### Audio Level Monitoring:
```c
// Monitor input audio levels
void monitor_audio_levels(uint8_t* buffer, uint32_t size) {
    int16_t max_level = 0;
    int16_t min_level = 32767;
    
    for (uint32_t i = 0; i < size; i += 2) {  // 16-bit samples
        int16_t sample = (int16_t)(buffer[i] | (buffer[i + 1] << 8));
        
        if (sample > max_level) max_level = sample;
        if (sample < min_level) min_level = sample;
    }
    
    // Convert to dBFS (decibels relative to full scale)
    float max_dB = 20.0f * log10f((float)abs(max_level) / 32767.0f);
    float min_dB = 20.0f * log10f((float)abs(min_level) / 32767.0f);
    
    printf("Audio Level: %.1f dBFS to %.1f dBFS\n", min_dB, max_dB);
}
```

### Recording Quality Considerations:
- **Sample Rate**: Higher rates capture more detail but use more bandwidth
- **Bit Depth**: Higher depth provides better dynamic range
- **Channels**: Mono uses half the bandwidth of stereo
- **Buffer Size**: Larger buffers prevent overflow but increase latency
- **Input Gain**: Proper gain prevents clipping and quantization noise

<a id='volume-control'></a>
## 7. Volume and Gain Control

### Volume Control Functions:
```c
// Set master volume (0-100%)
AUDIO_StatusTypeDef status = AUDIO_SetVolume(&haudio_sai, 75);
if (status != AUDIO_OK) {
    printf("Failed to set volume\n");
}

// Get current volume
uint8_t current_volume;
AUDIO_GetVolume(&haudio_sai, &current_volume);
printf("Current volume: %d%%\n", current_volume);
```

### Individual Channel Volume:
```c
// Set left channel volume
AUDIO_SetChannelVolume(&haudio_sai, AUDIO_CHANNEL_LEFT, 80);

// Set right channel volume
AUDIO_SetChannelVolume(&haudio_sai, AUDIO_CHANNEL_RIGHT, 60);

// Balance control
AUDIO_SetBalance(&haudio_sai, -20);  // -100 to +100 (left to right)
```

### Mute Control:
```c
// Mute audio output
AUDIO_Mute(&haudio_sai, true);

// Unmute audio output
AUDIO_Mute(&haudio_sai, false);

// Check mute status
bool is_muted;
AUDIO_IsMuted(&haudio_sai, &is_muted);
```

### Input Gain Control (Recording):
```c
// Set microphone gain
AUDIO_SetInputGain(&haudio_sai, AUDIO_INPUT_MIC, 50);  // 0-100%

// Set line input gain
AUDIO_SetInputGain(&haudio_sai, AUDIO_INPUT_LINE, 75);

// Automatic gain control
AUDIO_EnableAGC(&haudio_sai, true);
AUDIO_SetAGCLevel(&haudio_sai, 20);  // Target level in dB
```

### Volume Fading:
```c
// Fade volume over time
void fade_volume(uint8_t start_vol, uint8_t end_vol, uint32_t duration_ms) {
    uint32_t steps = 20;  // Number of fade steps
    uint32_t step_delay = duration_ms / steps;
    float vol_step = (float)(end_vol - start_vol) / steps;
    
    for (uint32_t i = 0; i <= steps; i++) {
        uint8_t current_vol = start_vol + (uint8_t)(vol_step * i);
        AUDIO_SetVolume(&haudio_sai, current_vol);
        HAL_Delay(step_delay);
    }
}

// Example: Fade in from silence to 80%
fade_volume(0, 80, 2000);  // 2-second fade in
```

### Digital Volume vs Analog Volume:
- **Digital Volume**: Applied in software, affects bit depth
- **Analog Volume**: Applied in codec hardware, preserves bit depth
- **Best Practice**: Use analog volume for main level, digital for fine tuning

### Volume Units:
- **Linear Scale**: 0-100% (what users expect)
- **dB Scale**: -60dB to +12dB (professional audio)
- **Codec Scale**: 0-255 (hardware specific)

### Volume Conversion:
```c
// Convert percentage to dB
float percent_to_db(uint8_t percent) {
    if (percent == 0) return -60.0f;  // Mute
    return 20.0f * log10f((float)percent / 100.0f);
}

// Convert dB to percentage
uint8_t db_to_percent(float db) {
    if (db <= -60.0f) return 0;
    return (uint8_t)(100.0f * powf(10.0f, db / 20.0f));
}
```

<a id='multi-channel'></a>
## 8. Multi-Channel Audio

### Multi-Channel Configuration:
```c
AUDIO_ConfigTypeDef multichannel_config = {
    .interface = AUDIO_INTERFACE_SAI,       // SAI supports TDM
    .sample_rate = AUDIO_SAMPLE_RATE_48K,
    .bit_depth = AUDIO_BIT_DEPTH_24,        // Higher quality
    .channels = AUDIO_CHANNELS_8,           // 8-channel TDM
    .volume = 80,
    .dma_enabled = true
};
```

### TDM (Time Division Multiplexing):
```c
// Configure TDM slots for 8-channel audio
AUDIO_TDMConfigTypeDef tdm_config = {
    .num_slots = 8,                         // 8 time slots
    .slot_size = AUDIO_SLOT_SIZE_32BIT,     // 32-bit slots
    .data_size = AUDIO_DATA_SIZE_24BIT,     // 24-bit data
    .frame_sync_width = 32,                 // Frame sync width
    .frame_length = 256                     // Total frame length
};

AUDIO_ConfigureTDM(&haudio_sai, &tdm_config);
```

### Multi-Channel Playback:
```c
// 8-channel audio buffer (24-bit samples)
#define CHANNELS 8
#define SAMPLES_PER_CHANNEL 1024
int32_t multichannel_buffer[CHANNELS * SAMPLES_PER_CHANNEL];

// Fill each channel with different content
for (int ch = 0; ch < CHANNELS; ch++) {
    for (int sample = 0; sample < SAMPLES_PER_CHANNEL; sample++) {
        // Generate different tones for each channel
        float frequency = 220.0f * (ch + 1);  // 220Hz, 440Hz, 660Hz, etc.
        float phase = 2.0f * 3.14159f * frequency * sample / 48000.0f;
        multichannel_buffer[ch * SAMPLES_PER_CHANNEL + sample] = 
            (int32_t)(8388607.0f * sinf(phase));  // 24-bit range
    }
}

// Play multi-channel audio
AUDIO_PlayMultichannel(&haudio_sai, (uint8_t*)multichannel_buffer, 
                      CHANNELS * SAMPLES_PER_CHANNEL * 4);  // 4 bytes per 24-bit sample
```

### Channel Routing and Mixing:
```c
// Route input channels to output channels
AUDIO_ChannelRouteTypeDef routing = {
    .input_channel = AUDIO_CHANNEL_0,
    .output_channel = AUDIO_CHANNEL_LEFT,
    .gain = 1.0f,                          // Unity gain
    .enable = true
};

AUDIO_SetChannelRouting(&haudio_sai, &routing);

// Mix multiple inputs to single output
AUDIO_MixChannels(&haudio_sai, 
                 AUDIO_CHANNEL_LEFT,       // Output channel
                 (AUDIO_ChannelTypeDef[]){AUDIO_CHANNEL_0, AUDIO_CHANNEL_1}, // Input channels
                 (float[]){0.7f, 0.3f},    // Mix ratios
                 2);                       // Number of inputs
```

### Surround Sound Setup:
```c
// 5.1 surround sound configuration
AUDIO_SurroundConfigTypeDef surround = {
    .front_left = AUDIO_CHANNEL_0,
    .front_right = AUDIO_CHANNEL_1,
    .center = AUDIO_CHANNEL_2,
    .subwoofer = AUDIO_CHANNEL_3,
    .rear_left = AUDIO_CHANNEL_4,
    .rear_right = AUDIO_CHANNEL_5,
    .enable_lfe = true,                    // Low frequency effects
    .lfe_gain = 0.8f
};

AUDIO_ConfigureSurround(&haudio_sai, &surround);
```

### Multi-Channel Recording:
```c
// Record from multiple microphones
#define MIC_CHANNELS 4
uint8_t mic_buffer[MIC_CHANNELS * 4096];  // 4KB per channel

AUDIO_StartMultichannelRecording(&haudio_sai, mic_buffer, 
                                MIC_CHANNELS * 4096, MIC_CHANNELS);

// Process each channel separately
for (int ch = 0; ch < MIC_CHANNELS; ch++) {
    uint8_t* channel_data = &mic_buffer[ch * 4096];
    process_microphone_data(channel_data, 4096, ch);
}
```

### Channel Considerations:
- **TDM Mode**: Required for more than 2 channels
- **Frame Sync**: Critical for channel synchronization
- **Buffer Alignment**: Data must be properly aligned for each channel
- **Processing Load**: Multi-channel increases CPU requirements
- **Memory Usage**: Larger buffers needed for multiple channels

<a id='dma-streaming'></a>
## 9. DMA Audio Streaming

### DMA Configuration:
```c
AUDIO_ConfigTypeDef dma_config = {
    .interface = AUDIO_INTERFACE_SAI,
    .sample_rate = AUDIO_SAMPLE_RATE_48K,
    .bit_depth = AUDIO_BIT_DEPTH_16,
    .channels = AUDIO_CHANNELS_STEREO,
    .volume = 80,
    .dma_enabled = true,                   // Enable DMA
    .buffer_mode = AUDIO_BUFFER_CIRCULAR   // Circular buffer
};
```

### Circular Buffer Streaming:
```c
#define AUDIO_BUFFER_SIZE 4096
uint8_t audio_buffer[AUDIO_BUFFER_SIZE];
volatile uint32_t buffer_offset = 0;

// Initialize DMA streaming
void init_audio_streaming(void) {
    AUDIO_ConfigTypeDef config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_48K,
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_STEREO,
        .dma_enabled = true,
        .buffer_mode = AUDIO_BUFFER_CIRCULAR
    };
    
    AUDIO_Init(&haudio_sai, &config);
    
    // Register callback for buffer half complete
    AUDIO_RegisterCallback(&haudio_sai, AUDIO_EVENT_HALF_COMPLETE, 
                          audio_half_callback);
    
    // Register callback for buffer complete
    AUDIO_RegisterCallback(&haudio_sai, AUDIO_EVENT_COMPLETE, 
                          audio_complete_callback);
}

// DMA callbacks
void audio_half_callback(AUDIO_HandleTypeDef* haudio) {
    // Fill first half of buffer
    fill_audio_buffer(&audio_buffer[0], AUDIO_BUFFER_SIZE / 2);
}

void audio_complete_callback(AUDIO_HandleTypeDef* haudio) {
    // Fill second half of buffer
    fill_audio_buffer(&audio_buffer[AUDIO_BUFFER_SIZE / 2], AUDIO_BUFFER_SIZE / 2);
}
```

### Double Buffering for Seamless Playback:
```c
// Double buffer implementation
#define BUFFER_COUNT 2
#define BUFFER_SIZE 2048
uint8_t audio_buffers[BUFFER_COUNT][BUFFER_SIZE];
volatile uint8_t active_buffer = 0;

// Fill buffer with audio data
void fill_audio_buffer(uint8_t* buffer, uint32_t size) {
    // Generate or load audio data
    for (uint32_t i = 0; i < size; i += 4) {  // Stereo 16-bit = 4 bytes per sample
        // Left channel
        int16_t left = generate_sample();
        buffer[i] = left & 0xFF;
        buffer[i + 1] = (left >> 8) & 0xFF;
        
        // Right channel
        int16_t right = generate_sample();
        buffer[i + 2] = right & 0xFF;
        buffer[i + 3] = (right >> 8) & 0xFF;
    }
}

// Start double-buffered playback
void start_double_buffer_playback(void) {
    // Fill both buffers initially
    fill_audio_buffer(audio_buffers[0], BUFFER_SIZE);
    fill_audio_buffer(audio_buffers[1], BUFFER_SIZE);
    
    // Start playback with first buffer
    AUDIO_PlayBuffer(&haudio_sai, audio_buffers[0], BUFFER_SIZE);
    
    // Set up DMA for continuous streaming
    AUDIO_EnableDMA(&haudio_sai, audio_buffers[active_buffer], BUFFER_SIZE);
}

void buffer_complete_callback(AUDIO_HandleTypeDef* haudio) {
    // Switch to next buffer
    active_buffer = (active_buffer + 1) % BUFFER_COUNT;
    
    // Fill the buffer that just finished playing
    fill_audio_buffer(audio_buffers[active_buffer], BUFFER_SIZE);
    
    // Continue with next buffer
    AUDIO_PlayBuffer(&haudio_sai, audio_buffers[active_buffer], BUFFER_SIZE);
}
```

### DMA for Recording:
```c
// Continuous recording with DMA
#define RECORD_BUFFER_SIZE 8192
uint8_t record_buffer[RECORD_BUFFER_SIZE];

void start_dma_recording(void) {
    AUDIO_ConfigTypeDef config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_16K,
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_MONO,
        .dma_enabled = true,
        .buffer_mode = AUDIO_BUFFER_CIRCULAR
    };
    
    AUDIO_Init(&haudio_sai, &config);
    
    // Start DMA recording
    AUDIO_StartRecording(&haudio_sai, record_buffer, RECORD_BUFFER_SIZE);
}

void recording_callback(AUDIO_HandleTypeDef* haudio) {
    // Process recorded data in real-time
    static uint32_t process_offset = 0;
    
    // Process latest chunk
    process_audio_chunk(&record_buffer[process_offset], RECORD_BUFFER_SIZE / 2);
    
    // Update offset for circular buffer
    process_offset = (process_offset + RECORD_BUFFER_SIZE / 2) % RECORD_BUFFER_SIZE;
}
```

### DMA Advantages:
- **CPU Free**: DMA handles data transfer automatically
- **Low Latency**: Minimal delay between audio events
- **High Bandwidth**: Supports high sample rates and bit depths
- **Continuous Streaming**: Seamless audio playback/recording
- **Buffer Management**: Automatic buffer switching

### DMA Considerations:
- **Memory Usage**: Large buffers consume RAM
- **Setup Complexity**: More complex initialization
- **Interrupt Handling**: Proper callback management required
- **Synchronization**: Buffer switching must be timed correctly
- **Debugging**: DMA issues can be hard to diagnose

<a id='audio-processing'></a>
## 10. Audio Processing

### Basic Filtering:
```c
// Simple low-pass filter
typedef struct {
    float alpha;        // Filter coefficient (0.0 to 1.0)
    float prev_output;  // Previous output sample
} LowPassFilter;

void init_lowpass_filter(LowPassFilter* filter, float cutoff_freq, float sample_rate) {
    // Calculate filter coefficient
    float rc = 1.0f / (2.0f * 3.14159f * cutoff_freq);
    float dt = 1.0f / sample_rate;
    filter->alpha = dt / (rc + dt);
    filter->prev_output = 0.0f;
}

int16_t process_lowpass_filter(LowPassFilter* filter, int16_t input) {
    // Apply filter: output = alpha * input + (1-alpha) * prev_output
    float output = filter->alpha * input + (1.0f - filter->alpha) * filter->prev_output;
    filter->prev_output = output;
    return (int16_t)output;
}

// Usage
LowPassFilter lpf;
init_lowpass_filter(&lpf, 1000.0f, 48000.0f);  // 1kHz cutoff at 48kHz

int16_t filtered_sample = process_lowpass_filter(&lpf, raw_sample);
```

### Echo/Delay Effect:
```c
#define DELAY_BUFFER_SIZE 48000  // 1 second at 48kHz
typedef struct {
    int16_t buffer[DELAY_BUFFER_SIZE];
    uint32_t write_index;
    uint32_t read_index;
    float feedback;     // 0.0 to 1.0
    float wet_dry;      // 0.0 (dry) to 1.0 (wet)
} DelayEffect;

void init_delay_effect(DelayEffect* delay, uint32_t delay_samples, float feedback, float wet_dry) {
    memset(delay->buffer, 0, sizeof(delay->buffer));
    delay->write_index = 0;
    delay->read_index = delay_samples;
    delay->feedback = feedback;
    delay->wet_dry = wet_dry;
}

int16_t process_delay_effect(DelayEffect* delay, int16_t input) {
    // Read delayed sample
    int16_t delayed = delay->buffer[delay->read_index];
    
    // Calculate output: mix dry and wet signals
    int16_t output = (int16_t)((1.0f - delay->wet_dry) * input + delay->wet_dry * delayed);
    
    // Write input + feedback to delay buffer
    delay->buffer[delay->write_index] = input + (int16_t)(delay->feedback * delayed);
    
    // Update indices
    delay->write_index = (delay->write_index + 1) % DELAY_BUFFER_SIZE;
    delay->read_index = (delay->read_index + 1) % DELAY_BUFFER_SIZE;
    
    return output;
}
```

### Audio Mixing:
```c
// Mix multiple audio streams
int16_t mix_audio_samples(int16_t sample1, int16_t sample2, float ratio1, float ratio2) {
    // Apply mixing ratios
    float mixed = ratio1 * sample1 + ratio2 * sample2;
    
    // Prevent clipping
    if (mixed > 32767.0f) mixed = 32767.0f;
    if (mixed < -32768.0f) mixed = -32768.0f;
    
    return (int16_t)mixed;
}

// Mix multiple channels
void mix_multichannel_audio(int16_t* output, int16_t** inputs, float* gains, 
                           uint32_t num_channels, uint32_t num_samples) {
    for (uint32_t sample = 0; sample < num_samples; sample++) {
        float mixed = 0.0f;
        
        for (uint32_t ch = 0; ch < num_channels; ch++) {
            mixed += gains[ch] * inputs[ch][sample];
        }
        
        // Apply compression/limiting if needed
        mixed = compressor_process(mixed);
        
        // Store result
        output[sample] = (int16_t)mixed;
    }
}
```

### Dynamic Range Compression:
```c
typedef struct {
    float threshold;    // Threshold in dB
    float ratio;        // Compression ratio
    float attack;       // Attack time in seconds
    float release;      // Release time in seconds
    float envelope;     // Current envelope
} Compressor;

void init_compressor(Compressor* comp, float threshold_db, float ratio, 
                    float attack_ms, float release_ms, float sample_rate) {
    comp->threshold = threshold_db;
    comp->ratio = ratio;
    comp->attack = 1.0f - expf(-1.0f / (attack_ms * 0.001f * sample_rate));
    comp->release = 1.0f - expf(-1.0f / (release_ms * 0.001f * sample_rate));
    comp->envelope = 0.0f;
}

int16_t process_compressor(Compressor* comp, int16_t input) {
    // Convert to dB
    float input_db = 20.0f * log10f(fabsf(input) / 32767.0f);
    
    // Calculate gain reduction
    float over_threshold = input_db - comp->threshold;
    float gain_reduction = 0.0f;
    
    if (over_threshold > 0.0f) {
        gain_reduction = over_threshold * (1.0f - 1.0f/comp->ratio);
    }
    
    // Smooth envelope
    if (gain_reduction > comp->envelope) {
        comp->envelope += comp->attack * (gain_reduction - comp->envelope);
    } else {
        comp->envelope += comp->release * (gain_reduction - comp->envelope);
    }
    
    // Apply gain reduction
    float output_db = input_db - comp->envelope;
    float output_linear = powf(10.0f, output_db / 20.0f);
    
    return (int16_t)(output_linear * 32767.0f * (input >= 0 ? 1.0f : -1.0f));
}
```

### Real-time Processing Pipeline:
```c
// Audio processing chain
typedef struct {
    LowPassFilter input_filter;
    DelayEffect echo;
    Compressor limiter;
    float master_gain;
} AudioProcessor;

void init_audio_processor(AudioProcessor* proc, float sample_rate) {
    // Initialize each effect
    init_lowpass_filter(&proc->input_filter, 15000.0f, sample_rate);  // High-pass
    init_delay_effect(&proc->echo, (uint32_t)(sample_rate * 0.3f), 0.4f, 0.3f);  // 300ms delay
    init_compressor(&proc->limiter, -6.0f, 4.0f, 5.0f, 100.0f, sample_rate);  // Compressor
    proc->master_gain = 0.8f;
}

int16_t process_audio_sample(AudioProcessor* proc, int16_t input) {
    float sample = input;
    
    // Apply processing chain
    sample = process_lowpass_filter(&proc->input_filter, sample);
    sample = process_delay_effect(&proc->echo, sample);
    sample = process_compressor(&proc->limiter, sample);
    sample *= proc->master_gain;
    
    // Prevent clipping
    if (sample > 32767.0f) sample = 32767.0f;
    if (sample < -32768.0f) sample = -32768.0f;
    
    return (int16_t)sample;
}
```

### Processing Considerations:
- **Latency**: Each effect adds processing delay
- **CPU Usage**: Complex effects require more processing power
- **Memory**: Delay buffers consume RAM
- **Precision**: Use floating point for intermediate calculations
- **Clipping**: Always prevent digital clipping

<a id='codec-integration'></a>
## 11. Codec Integration

### WM8994 Codec Configuration:
```c
// WM8994 codec handle
WM8994_HandleTypeDef hcodec;

// Initialize codec
void init_wm8994_codec(void) {
    hcodec.DeviceAddr = WM8994_ADDR_0;
    hcodec.OutputDevice = WM8994_OUT_HEADPHONE;
    hcodec.InputDevice = WM8994_IN_MIC1;
    hcodec.Frequency = WM8994_FREQUENCY_48K;
    hcodec.Resolution = WM8994_RESOLUTION_16B;
    hcodec.Volume = 80;
    
    // Initialize codec hardware
    if (WM8994_Init(&hcodec, &hi2c1) != WM8994_OK) {
        printf("Codec initialization failed\n");
        return;
    }
    
    // Configure audio interface
    WM8994_SetAudioInterface(&hcodec, WM8994_INTERFACE_I2S);
    
    // Set volume levels
    WM8994_SetVolume(&hcodec, WM8994_OUT_HEADPHONE, 80);
    WM8994_SetVolume(&hcodec, WM8994_IN_MIC1, 60);
}
```

### Codec Control Functions:
```c
// Set codec output device
WM8994_SetOutputMode(&hcodec, WM8994_OUT_HEADPHONE);  // Headphones
WM8994_SetOutputMode(&hcodec, WM8994_OUT_SPEAKER);    // Speaker

// Set codec input device
WM8994_SetInputMode(&hcodec, WM8994_IN_MIC1);         // Microphone 1
WM8994_SetInputMode(&hcodec, WM8994_IN_LINE1);        // Line input

// Control codec power
WM8994_PowerUp(&hcodec);                              // Power on
WM8994_PowerDown(&hcodec);                            // Power off
```

### Audio Path Configuration:
```c
// Configure playback path: DAC -> Headphone amplifier
WM8994_Play(&hcodec, WM8994_OUT_HEADPHONE);

// Configure recording path: Microphone -> ADC
WM8994_Record(&hcodec, WM8994_IN_MIC1);

// Configure bypass path: Line in -> Line out
WM8994_SetPath(&hcodec, WM8994_PATH_BYPASS);
```

### Codec Register Access:
```c
// Read codec register
uint16_t reg_value;
WM8994_ReadRegister(&hcodec, WM8994_REG_LEFT_LINE_INPUT_1_2_VOLUME, &reg_value);

// Write codec register
WM8994_WriteRegister(&hcodec, WM8994_REG_LEFT_OUTPUT_VOLUME, 0x50);

// Configure equalizer
WM8994_EnableEQ(&hcodec, true);
WM8994_SetEQBand(&hcodec, WM8994_EQ_BAND_BASS, 12);    // +12dB bass boost
WM8994_SetEQBand(&hcodec, WM8994_EQ_BAND_TREBLE, -6);  // -6dB treble cut
```

### Codec Synchronization:
```c
// Ensure codec and SAI are synchronized
void synchronize_codec_and_sai(void) {
    // Configure SAI first
    AUDIO_ConfigTypeDef sai_config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_48K,
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_STEREO
    };
    AUDIO_Init(&haudio_sai, &sai_config);
    
    // Configure codec to match
    hcodec.Frequency = WM8994_FREQUENCY_48K;
    hcodec.Resolution = WM8994_RESOLUTION_16B;
    WM8994_Init(&hcodec, &hi2c1);
    
    // Start both simultaneously
    AUDIO_StartPlayback(&haudio_sai);
    WM8994_Play(&hcodec, WM8994_OUT_HEADPHONE);
}
```

### Multiple Codec Support:
```c
// Support for multiple codecs on same bus
#define CODEC_COUNT 2
WM8994_HandleTypeDef codecs[CODEC_COUNT];

void init_multiple_codecs(void) {
    // Codec 1 - Address 0x1A
    codecs[0].DeviceAddr = WM8994_ADDR_0;
    WM8994_Init(&codecs[0], &hi2c1);
    
    // Codec 2 - Address 0x1B (if available)
    codecs[1].DeviceAddr = WM8994_ADDR_1;
    WM8994_Init(&codecs[1], &hi2c1);
    
    // Configure different outputs
    WM8994_SetOutputMode(&codecs[0], WM8994_OUT_HEADPHONE);
    WM8994_SetOutputMode(&codecs[1], WM8994_OUT_SPEAKER);
}
```

### Codec Diagnostics:
```c
// Check codec status
void diagnose_codec(void) {
    uint16_t reg_val;
    
    // Check codec ID
    WM8994_ReadRegister(&hcodec, WM8994_REG_SOFTWARE_RESET, &reg_val);
    printf("Codec ID: 0x%04X (expected: 0x8994)\n", reg_val);
    
    // Check PLL lock status
    WM8994_ReadRegister(&hcodec, WM8994_REG_PLL1_K_3, &reg_val);
    if (reg_val & 0x8000) {
        printf("PLL locked\n");
    } else {
        printf("PLL not locked - check clock input\n");
    }
    
    // Check power status
    WM8994_ReadRegister(&hcodec, WM8994_REG_POWER_MANAGEMENT_1, &reg_val);
    printf("Power status: 0x%04X\n", reg_val);
}
```

### Codec Integration Best Practices:
- **Initialization Order**: Configure SAI first, then codec
- **Clock Synchronization**: Ensure MCLK is stable before codec init
- **Power Sequencing**: Follow codec power-up/power-down sequence
- **Register Caching**: Cache frequently read registers
- **Error Recovery**: Implement codec reset on communication errors
- **Temperature Compensation**: Monitor and compensate for temperature effects

<a id='advanced'></a>
## 12. Advanced Features

### Interrupt-Driven Audio:
```c
// Register audio event callbacks
void audio_event_callback(AUDIO_HandleTypeDef* haudio, AUDIO_EventTypeDef event) {
    switch (event) {
        case AUDIO_EVENT_HALF_COMPLETE:
            // Fill first half of DMA buffer
            fill_audio_buffer(&audio_buffer[0], BUFFER_SIZE / 2);
            break;
            
        case AUDIO_EVENT_COMPLETE:
            // Fill second half of DMA buffer
            fill_audio_buffer(&audio_buffer[BUFFER_SIZE / 2], BUFFER_SIZE / 2);
            break;
            
        case AUDIO_EVENT_ERROR:
            // Handle audio error
            AUDIO_ErrorHandler(haudio);
            break;
            
        case AUDIO_EVENT_OVERFLOW:
            // Handle buffer overflow
            AUDIO_StopPlayback(haudio);
            AUDIO_StartPlayback(haudio);  // Restart
            break;
    }
}

// Register callbacks
AUDIO_RegisterCallback(&haudio_sai, AUDIO_EVENT_ALL, audio_event_callback);
```

### Audio Statistics and Monitoring:
```c
// Get audio performance statistics
AUDIO_StatsTypeDef stats;
AUDIO_GetStatistics(&haudio_sai, &stats);

printf("Audio Statistics:\n");
printf("Total samples played: %lu\n", stats.samples_played);
printf("Buffer underruns: %lu\n", stats.underruns);
printf("Buffer overruns: %lu\n", stats.overruns);
printf("Average CPU usage: %.1f%%\n", stats.cpu_usage);

// Monitor audio levels
AUDIO_LevelTypeDef levels;
AUDIO_GetLevels(&haudio_sai, &levels);

printf("Audio Levels:\n");
printf("Left peak: %.1f dBFS\n", levels.left_peak);
printf("Right peak: %.1f dBFS\n", levels.right_peak);
printf("RMS level: %.1f dBFS\n", levels.rms_level);
```

### Audio Format Conversion:
```c
// Convert between different audio formats
void convert_audio_format(uint8_t* input, uint8_t* output, 
                         AUDIO_FormatTypeDef input_format, 
                         AUDIO_FormatTypeDef output_format,
                         uint32_t num_samples) {
    
    for (uint32_t i = 0; i < num_samples; i++) {
        // Read input sample
        int32_t sample = 0;
        
        switch (input_format) {
            case AUDIO_FORMAT_16BIT:
                sample = (int16_t)(input[i*2] | (input[i*2+1] << 8));
                break;
            case AUDIO_FORMAT_24BIT:
                sample = (int32_t)(input[i*3] | (input[i*3+1] << 8) | (input[i*3+2] << 16));
                if (sample & 0x800000) sample |= 0xFF000000;  // Sign extend
                break;
            case AUDIO_FORMAT_32BIT:
                sample = (int32_t)(input[i*4] | (input[i*4+1] << 8) | 
                                   (input[i*4+2] << 16) | (input[i*4+3] << 24));
                break;
        }
        
        // Write output sample
        switch (output_format) {
            case AUDIO_FORMAT_16BIT:
                output[i*2] = sample & 0xFF;
                output[i*2+1] = (sample >> 8) & 0xFF;
                break;
            case AUDIO_FORMAT_24BIT:
                output[i*3] = sample & 0xFF;
                output[i*3+1] = (sample >> 8) & 0xFF;
                output[i*3+2] = (sample >> 16) & 0xFF;
                break;
            case AUDIO_FORMAT_32BIT:
                output[i*4] = sample & 0xFF;
                output[i*4+1] = (sample >> 8) & 0xFF;
                output[i*4+2] = (sample >> 16) & 0xFF;
                output[i*4+3] = (sample >> 24) & 0xFF;
                break;
        }
    }
}
```

### Audio Streaming over Network:
```c
// Stream audio over Ethernet/UDP
#define AUDIO_STREAM_PORT 50000
#define PACKET_SIZE 1024

void init_audio_streaming(void) {
    // Initialize network interface
    // Initialize UDP socket
    
    // Configure audio for streaming
    AUDIO_ConfigTypeDef config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_16K,  // Lower rate for network
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_MONO,
        .dma_enabled = true
    };
    
    AUDIO_Init(&haudio_sai, &config);
    AUDIO_StartRecording(&haudio_sai, stream_buffer, PACKET_SIZE);
}

void stream_audio_packet(uint8_t* data, uint32_t size) {
    // Compress audio if needed
    // Encrypt if required
    // Send over network
    
    UDP_SendPacket(audio_socket, data, size);
}
```

### Power Management:
```c
// Audio power management
void enter_audio_standby(void) {
    // Mute outputs
    AUDIO_Mute(&haudio_sai, true);
    
    // Stop audio processing
    AUDIO_StopPlayback(&haudio_sai);
    AUDIO_StopRecording(&haudio_sai);
    
    // Power down codec
    WM8994_PowerDown(&hcodec);
    
    // Disable SAI peripheral
    AUDIO_DeInit(&haudio_sai);
}

void exit_audio_standby(void) {
    // Reinitialize audio system
    AUDIO_Init(&haudio_sai, &audio_config);
    WM8994_Init(&hcodec, &hi2c1);
    
    // Fade in volume
    AUDIO_Mute(&haudio_sai, false);
    fade_volume(0, 80, 500);
}
```

### Audio Testing and Calibration:
```c
// Generate test signals
void generate_test_signal(AUDIO_TestSignalTypeDef signal_type, 
                         uint8_t* buffer, uint32_t size) {
    
    for (uint32_t i = 0; i < size; i += 2) {
        int16_t sample = 0;
        
        switch (signal_type) {
            case AUDIO_TEST_SINE:
                sample = (int16_t)(16384 * sinf(2 * 3.14159f * 1000 * i / 48000));
                break;
            case AUDIO_TEST_SQUARE:
                sample = (i % (48000 / 1000) < 24000 / 1000) ? 16384 : -16384;
                break;
            case AUDIO_TEST_NOISE:
                sample = rand() % 32768 - 16384;
                break;
            case AUDIO_TEST_SWEEP:
                // Frequency sweep implementation
                break;
        }
        
        buffer[i] = sample & 0xFF;
        buffer[i+1] = (sample >> 8) & 0xFF;
    }
}

// Audio calibration
void calibrate_audio_system(void) {
    // Generate 1kHz test tone at -20dBFS
    generate_test_signal(AUDIO_TEST_SINE, cal_buffer, CAL_BUFFER_SIZE);
    
    // Play calibration tone
    AUDIO_PlayBuffer(&haudio_sai, cal_buffer, CAL_BUFFER_SIZE);
    
    // Measure output level with external equipment
    // Adjust codec gain registers accordingly
    
    printf("Adjust output level to -20dBFS and press enter\n");
    getchar();
    
    // Store calibration values in non-volatile memory
    save_calibration_data();
}
```

### Advanced Audio Processing:
```c
// Real-time FFT analysis
#define FFT_SIZE 1024
float fft_input[FFT_SIZE];
float fft_output[FFT_SIZE];

void perform_fft_analysis(int16_t* audio_samples, uint32_t num_samples) {
    // Convert to float and apply window
    for (uint32_t i = 0; i < num_samples; i++) {
        fft_input[i] = (float)audio_samples[i] / 32768.0f;
        fft_input[i] *= 0.5f * (1.0f - cosf(2.0f * 3.14159f * i / (num_samples - 1)));  // Hann window
    }
    
    // Perform FFT (using CMSIS-DSP library)
    arm_rfft_fast_f32(&fft_instance, fft_input, fft_output, 0);
    
    // Calculate magnitude spectrum
    for (uint32_t i = 0; i < num_samples / 2; i++) {
        float real = fft_output[2 * i];
        float imag = fft_output[2 * i + 1];
        float magnitude = sqrtf(real * real + imag * imag);
        
        // Use magnitude for spectrum analysis, equalization, etc.
        process_frequency_bin(i, magnitude);
    }
}
```

### Performance Optimization:
- **Use DMA** for all high-bandwidth audio operations
- **Optimize buffer sizes** based on latency requirements
- **Implement circular buffers** for continuous streaming
- **Minimize interrupt service routines**
- **Use appropriate sample rates** for the application
- **Profile audio processing** to identify bottlenecks
- **Consider cache alignment** for audio buffers

<a id='troubleshooting'></a>
## 13. Troubleshooting

### Common Issues and Solutions:

#### 1. No Audio Output
**Symptoms:** Audio plays but no sound from speakers/headphones
**Possible Causes:**
- Codec not initialized or configured incorrectly
- Wrong output device selected (headphone vs speaker)
- Muted outputs or zero volume
- SAI pins not configured correctly
- Codec power issues
**Solutions:**
- Check codec initialization status
- Verify output device selection with `WM8994_SetOutputMode()`
- Check volume levels with `AUDIO_GetVolume()`
- Verify SAI pin configuration in STM32CubeMX
- Check codec power supply and I2C communication

#### 2. Distorted Audio
**Symptoms:** Audio sounds distorted or crackly
**Possible Causes:**
- Buffer underruns (DMA too slow)
- Incorrect sample rate/bit depth mismatch
- Codec configuration errors
- Power supply noise
- Ground loops
**Solutions:**
- Increase buffer sizes for DMA
- Verify sample rate/bit depth consistency between SAI and codec
- Check codec register settings
- Add decoupling capacitors
- Improve grounding and shielding

#### 3. Recording Issues
**Symptoms:** Cannot record audio or poor quality recordings
**Possible Causes:**
- Wrong input device selected
- Microphone gain too low/high
- Incorrect SAI configuration for input
- Codec ADC not enabled
- Input signal too weak
**Solutions:**
- Check input device selection with `WM8994_SetInputMode()`
- Adjust microphone gain with `AUDIO_SetInputGain()`
- Verify SAI configuration for recording
- Enable codec ADC channels
- Check input signal levels

#### 4. Synchronization Problems
**Symptoms:** Audio glitches or timing issues
**Possible Causes:**
- SAI and codec clock mismatch
- DMA buffer management issues
- Interrupt timing problems
- PLL not locked
**Solutions:**
- Ensure SAI and codec use same sample rate
- Check DMA buffer sizes and alignment
- Verify interrupt priorities
- Check PLL status and clock inputs

#### 5. High CPU Usage
**Symptoms:** System becomes unresponsive during audio processing
**Possible Causes:**
- Polling mode instead of DMA
- Complex audio processing algorithms
- Small buffer sizes causing frequent interrupts
- Inefficient code in audio callbacks
**Solutions:**
- Enable DMA for all audio operations
- Optimize audio processing algorithms
- Increase buffer sizes to reduce interrupt frequency
- Profile and optimize callback functions

#### 6. I2C Communication Errors
**Symptoms:** Codec initialization fails
**Possible Causes:**
- Wrong I2C address
- I2C bus issues (pull-ups, speed)
- Codec not powered or reset
- Pin conflicts
**Solutions:**
- Verify codec I2C address (0x1A or 0x1B)
- Check I2C pull-up resistors and bus speed
- Ensure proper power sequencing
- Check for pin conflicts in STM32CubeMX

#### 7. DMA Transfer Errors
**Symptoms:** Audio stuttering or dropouts
**Possible Causes:**
- DMA channel conflicts
- Incorrect DMA configuration
- Memory alignment issues
- Buffer size mismatches
**Solutions:**
- Check DMA channel allocation
- Verify DMA stream and channel configuration
- Ensure buffers are properly aligned
- Match buffer sizes between DMA and audio

### Debug Tips:
```c
// Check audio system status
AUDIO_StatusTypeDef status = AUDIO_GetStatus(&haudio_sai);
printf("Audio Status: %s\n", AUDIO_GetStatusString(status));

// Verify codec communication
uint16_t codec_id;
if (WM8994_ReadRegister(&hcodec, WM8994_REG_SOFTWARE_RESET, &codec_id) == WM8994_OK) {
    printf("Codec ID: 0x%04X (expected: 0x8994)\n", codec_id);
} else {
    printf("Codec communication failed\n");
}

// Check SAI registers
printf("SAI GCR: 0x%08X\n", SAI1->GCR);
printf("SAI CR1: 0x%08X\n", SAI1_Block_A->CR1);
printf("SAI SR: 0x%08X\n", SAI1_Block_A->SR);

// Monitor audio buffer status
AUDIO_BufferStatusTypeDef buf_status;
AUDIO_GetBufferStatus(&haudio_sai, &buf_status);
printf("Buffer usage: %d%%\n", buf_status.usage_percent);

// Test basic audio functionality
uint8_t test_buffer[1024];
memset(test_buffer, 0x80, sizeof(test_buffer));  // Middle value
if (AUDIO_PlayBuffer(&haudio_sai, test_buffer, sizeof(test_buffer)) == AUDIO_OK) {
    printf("Basic playback test passed\n");
} else {
    printf("Basic playback test failed\n");
}
```

### Performance Optimization Tips:
- **Use DMA** for all audio data transfer
- **Optimize buffer sizes** based on latency requirements
- **Implement circular buffers** for continuous streaming
- **Minimize interrupt service routines**
- **Use appropriate sample rates** for the application
- **Profile audio processing** to identify bottlenecks
- **Consider cache alignment** for audio buffers

### Hardware Debugging:
- **Oscilloscope**: Check SAI signals (MCK, SCK, FS, SD)
- **Logic Analyzer**: Verify I2S/SAI protocol compliance
- **Audio Analyzer**: Measure frequency response and distortion
- **Power Supply**: Monitor voltage levels and ripple
- **Spectrum Analyzer**: Check for noise and interference

### Getting Help:
- Check STM32F429 Reference Manual (SAI chapter)
- Review codec datasheet (WM8994 or equivalent)
- Verify HAL driver implementation
- Test with known working hardware
- Check audio forum communities for similar issues

For more detailed information, refer to the STM32F429 datasheet, SAI/I2S specifications, and codec reference manuals.

## Complete Example Program

```c
#include "audio.h"
#include "wm8994.h"
#include "stm32f4xx.h"
#include <stdio.h>
#include <math.h>

// Audio configuration
#define AUDIO_SAMPLE_RATE    48000
#define AUDIO_BUFFER_SIZE    4096
#define AUDIO_CHANNELS       2

// Global handles
AUDIO_HandleTypeDef haudio;
WM8994_HandleTypeDef hcodec;
I2C_HandleTypeDef hi2c1;

// Audio buffers
uint8_t audio_buffer[AUDIO_BUFFER_SIZE];
volatile uint32_t buffer_offset = 0;

// Sine wave generation for testing
float phase = 0.0f;
const float frequency = 440.0f;  // A4 note

void generate_sine_wave(uint8_t* buffer, uint32_t size) {
    for (uint32_t i = 0; i < size; i += 4) {  // Stereo 16-bit = 4 bytes
        // Generate sine wave sample
        int16_t sample = (int16_t)(16384 * sinf(phase));
        phase += 2.0f * 3.14159f * frequency / AUDIO_SAMPLE_RATE;
        
        if (phase >= 2.0f * 3.14159f) {
            phase -= 2.0f * 3.14159f;
        }
        
        // Left channel
        buffer[i] = sample & 0xFF;
        buffer[i + 1] = (sample >> 8) & 0xFF;
        
        // Right channel (same as left for mono sound)
        buffer[i + 2] = sample & 0xFF;
        buffer[i + 3] = (sample >> 8) & 0xFF;
    }
}

void audio_half_callback(AUDIO_HandleTypeDef* haudio) {
    // Fill first half of buffer
    generate_sine_wave(&audio_buffer[0], AUDIO_BUFFER_SIZE / 2);
}

void audio_complete_callback(AUDIO_HandleTypeDef* haudio) {
    // Fill second half of buffer
    generate_sine_wave(&audio_buffer[AUDIO_BUFFER_SIZE / 2], AUDIO_BUFFER_SIZE / 2);
}

int main(void) {
    printf("STM32F429 Audio Demonstration\n\n");
    
    // Initialize I2C for codec communication
    hi2c1.Instance = I2C1;
    hi2c1.Init.ClockSpeed = 100000;
    hi2c1.Init.DutyCycle = I2C_DUTYCYCLE_2;
    hi2c1.Init.OwnAddress1 = 0;
    hi2c1.Init.AddressingMode = I2C_ADDRESSINGMODE_7BIT;
    hi2c1.Init.DualAddressMode = I2C_DUALADDRESS_DISABLE;
    hi2c1.Init.OwnAddress2 = 0;
    hi2c1.Init.GeneralCallMode = I2C_GENERALCALL_DISABLE;
    hi2c1.Init.NoStretchMode = I2C_NOSTRETCH_DISABLE;
    
    if (HAL_I2C_Init(&hi2c1) != HAL_OK) {
        printf("I2C initialization failed\n");
        return -1;
    }
    
    // Initialize codec
    hcodec.DeviceAddr = WM8994_ADDR_0;
    hcodec.OutputDevice = WM8994_OUT_HEADPHONE;
    hcodec.InputDevice = WM8994_IN_NONE;
    hcodec.Frequency = WM8994_FREQUENCY_48K;
    hcodec.Resolution = WM8994_RESOLUTION_16B;
    hcodec.Volume = 80;
    
    if (WM8994_Init(&hcodec, &hi2c1) != WM8994_OK) {
        printf("Codec initialization failed\n");
        return -1;
    }
    printf("Codec initialized successfully\n");
    
    // Configure audio system
    AUDIO_ConfigTypeDef audio_config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_48K,
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_STEREO,
        .volume = 70,
        .dma_enabled = true,
        .buffer_mode = AUDIO_BUFFER_CIRCULAR
    };
    
    AUDIO_StatusTypeDef status = AUDIO_Init(&haudio, &audio_config);
    if (status != AUDIO_OK) {
        printf("Audio initialization failed: %s\n", AUDIO_GetStatusString(status));
        return -1;
    }
    printf("Audio system initialized successfully\n\n");
    
    // Register audio callbacks
    AUDIO_RegisterCallback(&haudio, AUDIO_EVENT_HALF_COMPLETE, audio_half_callback);
    AUDIO_RegisterCallback(&haudio, AUDIO_EVENT_COMPLETE, audio_complete_callback);
    
    // Fill initial buffer
    generate_sine_wave(audio_buffer, AUDIO_BUFFER_SIZE);
    
    // Start audio playback
    status = AUDIO_StartPlayback(&haudio);
    if (status != AUDIO_OK) {
        printf("Failed to start playback: %s\n", AUDIO_GetStatusString(status));
        return -1;
    }
    printf("Audio playback started - you should hear a 440Hz tone\n");
    printf("Press any key to stop...\n");
    
    // Wait for user input
    getchar();
    
    // Stop playback
    AUDIO_StopPlayback(&haudio);
    printf("Audio playback stopped\n");
    
    // Demonstrate recording
    printf("\nDemonstrating audio recording...\n");
    
    uint8_t record_buffer[2048];
    status = AUDIO_StartRecording(&haudio, record_buffer, sizeof(record_buffer));
    if (status == AUDIO_OK) {
        printf("Recording for 2 seconds...\n");
        HAL_Delay(2000);
        AUDIO_StopRecording(&haudio);
        printf("Recording complete\n");
        
        // Process recorded data (simple level detection)
        int16_t max_level = 0;
        for (uint32_t i = 0; i < sizeof(record_buffer); i += 2) {
            int16_t sample = (int16_t)(record_buffer[i] | (record_buffer[i + 1] << 8));
            if (abs(sample) > max_level) max_level = abs(sample);
        }
        
        float level_db = 20.0f * log10f((float)max_level / 32767.0f);
        printf("Recorded audio peak level: %.1f dBFS\n", level_db);
    }
    
    printf("\nAudio demonstration complete!\n");
    
    return 0;
}
```

## Summary

This tutorial covered:
- Audio system fundamentals and STM32F429 audio capabilities
- Hardware setup with SAI/I2S interfaces and codec integration
- Basic audio playback and recording operations
- Volume control and gain management
- Multi-channel audio and surround sound
- DMA streaming for high-performance audio
- Audio processing (filtering, effects, mixing)
- Codec integration with WM8994
- Advanced features (interrupts, statistics, format conversion)
- Troubleshooting common audio issues

### Key Takeaways:
1. **Always synchronize** SAI and codec configurations
2. **Use DMA** for continuous audio streaming
3. **Check return values** from all audio functions
4. **Implement proper error handling** for robust operation
5. **Consider buffer sizes** for latency vs memory trade-offs
6. **Test incrementally** when setting up audio systems

### Next Steps:
- Experiment with different sample rates and bit depths
- Implement audio effects and processing algorithms
- Add network streaming capabilities
- Integrate with audio analysis libraries
- Develop custom audio applications

### Additional Resources:
- STM32F429 Reference Manual (SAI chapter)
- WM8994 Codec Datasheet
- I2S and SAI protocol specifications
- Audio processing algorithm references
- Digital signal processing tutorials

For more detailed information, refer to the STM32F429 datasheet, audio codec datasheets, and digital audio standards documentation.

## Complete Example Program

```c
#include "audio.h"
#include "wm8994.h"
#include "stm32f4xx.h"
#include <stdio.h>
#include <math.h>

// Audio configuration
#define AUDIO_SAMPLE_RATE    48000
#define AUDIO_BUFFER_SIZE    4096
#define AUDIO_CHANNELS       2

// Global handles
AUDIO_HandleTypeDef haudio;
WM8994_HandleTypeDef hcodec;
I2C_HandleTypeDef hi2c1;

// Audio buffers
uint8_t audio_buffer[AUDIO_BUFFER_SIZE];
volatile uint32_t buffer_offset = 0;

// Sine wave generation for testing
float phase = 0.0f;
const float frequency = 440.0f;  // A4 note

void generate_sine_wave(uint8_t* buffer, uint32_t size) {
    for (uint32_t i = 0; i < size; i += 4) {  // Stereo 16-bit = 4 bytes
        // Generate sine wave sample
        int16_t sample = (int16_t)(16384 * sinf(phase));
        phase += 2.0f * 3.14159f * frequency / AUDIO_SAMPLE_RATE;
        
        if (phase >= 2.0f * 3.14159f) {
            phase -= 2.0f * 3.14159f;
        }
        
        // Left channel
        buffer[i] = sample & 0xFF;
        buffer[i + 1] = (sample >> 8) & 0xFF;
        
        // Right channel (same as left for mono sound)
        buffer[i + 2] = sample & 0xFF;
        buffer[i + 3] = (sample >> 8) & 0xFF;
    }
}

void audio_half_callback(AUDIO_HandleTypeDef* haudio) {
    // Fill first half of buffer
    generate_sine_wave(&audio_buffer[0], AUDIO_BUFFER_SIZE / 2);
}

void audio_complete_callback(AUDIO_HandleTypeDef* haudio) {
    // Fill second half of buffer
    generate_sine_wave(&audio_buffer[AUDIO_BUFFER_SIZE / 2], AUDIO_BUFFER_SIZE / 2);
}

int main(void) {
    printf("STM32F429 Audio Demonstration\n\n");
    
    // Initialize I2C for codec communication
    hi2c1.Instance = I2C1;
    hi2c1.Init.ClockSpeed = 100000;
    hi2c1.Init.DutyCycle = I2C_DUTYCYCLE_2;
    hi2c1.Init.OwnAddress1 = 0;
    hi2c1.Init.AddressingMode = I2C_ADDRESSINGMODE_7BIT;
    hi2c1.Init.DualAddressMode = I2C_DUALADDRESS_DISABLE;
    hi2c1.Init.OwnAddress2 = 0;
    hi2c1.Init.GeneralCallMode = I2C_GENERALCALL_DISABLE;
    hi2c1.Init.NoStretchMode = I2C_NOSTRETCH_DISABLE;
    
    if (HAL_I2C_Init(&hi2c1) != HAL_OK) {
        printf("I2C initialization failed\n");
        return -1;
    }
    
    // Initialize codec
    hcodec.DeviceAddr = WM8994_ADDR_0;
    hcodec.OutputDevice = WM8994_OUT_HEADPHONE;
    hcodec.InputDevice = WM8994_IN_NONE;
    hcodec.Frequency = WM8994_FREQUENCY_48K;
    hcodec.Resolution = WM8994_RESOLUTION_16B;
    hcodec.Volume = 80;
    
    if (WM8994_Init(&hcodec, &hi2c1) != WM8994_OK) {
        printf("Codec initialization failed\n");
        return -1;
    }
    printf("Codec initialized successfully\n");
    
    // Configure audio system
    AUDIO_ConfigTypeDef audio_config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_48K,
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_STEREO,
        .volume = 70,
        .dma_enabled = true,
        .buffer_mode = AUDIO_BUFFER_CIRCULAR
    };
    
    AUDIO_StatusTypeDef status = AUDIO_Init(&haudio, &audio_config);
    if (status != AUDIO_OK) {
        printf("Audio initialization failed: %s\n", AUDIO_GetStatusString(status));
        return -1;
    }
    printf("Audio system initialized successfully\n\n");
    
    // Register audio callbacks
    AUDIO_RegisterCallback(&haudio, AUDIO_EVENT_HALF_COMPLETE, audio_half_callback);
    AUDIO_RegisterCallback(&haudio, AUDIO_EVENT_COMPLETE, audio_complete_callback);
    
    // Fill initial buffer
    generate_sine_wave(audio_buffer, AUDIO_BUFFER_SIZE);
    
    // Start audio playback
    status = AUDIO_StartPlayback(&haudio);
    if (status != AUDIO_OK) {
        printf("Failed to start playback: %s\n", AUDIO_GetStatusString(status));
        return -1;
    }
    printf("Audio playback started - you should hear a 440Hz tone\n");
    printf("Press any key to stop...\n");
    
    // Wait for user input
    getchar();
    
    // Stop playback
    AUDIO_StopPlayback(&haudio);
    printf("Audio playback stopped\n");
    
    // Demonstrate recording
    printf("\nDemonstrating audio recording...\n");
    
    uint8_t record_buffer[2048];
    status = AUDIO_StartRecording(&haudio, record_buffer, sizeof(record_buffer));
    if (status == AUDIO_OK) {
        printf("Recording for 2 seconds...\n");
        HAL_Delay(2000);
        AUDIO_StopRecording(&haudio);
        printf("Recording complete\n");
        
        // Process recorded data (simple level detection)
        int16_t max_level = 0;
        for (uint32_t i = 0; i < sizeof(record_buffer); i += 2) {
            int16_t sample = (int16_t)(record_buffer[i] | (record_buffer[i + 1] << 8));
            if (abs(sample) > max_level) max_level = abs(sample);
        }
        
        float level_db = 20.0f * log10f((float)max_level / 32767.0f);
        printf("Recorded audio peak level: %.1f dBFS\n", level_db);
    }
    
    printf("\nAudio demonstration complete!\n");
    
    return 0;
}
```

## Summary

This tutorial covered:
- Audio system fundamentals and STM32F429 audio capabilities
- Hardware setup with SAI/I2S interfaces and codec integration
- Basic audio playback and recording operations
- Volume control and gain management
- Multi-channel audio and surround sound
- DMA streaming for high-performance audio
- Audio processing (filtering, effects, mixing)
- Codec integration with WM8994
- Advanced features (interrupts, statistics, format conversion)
- Troubleshooting common audio issues

### Key Takeaways:
1. **Always synchronize** SAI and codec configurations
2. **Use DMA** for continuous audio streaming
3. **Check return values** from all audio functions
4. **Implement proper error handling** for robust operation
5. **Consider buffer sizes** for latency vs memory trade-offs
6. **Test incrementally** when setting up audio systems

### Next Steps:
- Experiment with different sample rates and bit depths
- Implement audio effects and processing algorithms
- Add network streaming capabilities
- Integrate with audio analysis libraries
- Develop custom audio applications

### Additional Resources:
- STM32F429 Reference Manual (SAI chapter)
- WM8994 Codec Datasheet
- I2S and SAI protocol specifications
- Audio processing algorithm references
- Digital signal processing tutorials

For more detailed information, refer to the STM32F429 datasheet, audio codec datasheets, and digital audio standards documentation.

<a id='software-setup'></a>
## 4. Software Dependencies

### Required Files:
- `audio.h` - Header file with function prototypes and types
- `audio.c` - Main audio driver implementation
- STM32F4xx HAL Library (`stm32f4xx_hal.h`, `stm32f4xx_hal_sai.h`, `stm32f4xx_hal_i2s.h`)
- Standard C libraries: `stdint.h`, `stdbool.h`, `string.h`, `stdlib.h`

### Include Path Setup:
```c
#include "audio.h"
#include "stm32f4xx.h"  // HAL library
```

### Build Configuration:
- Add source files to project: `audio.c`
- Include header paths: path to `audio.h`
- Enable SAI/I2S peripheral in STM32CubeMX
- Configure audio PLL for proper clocking
- Enable DMA for audio streaming
- Configure I2C for codec control

### Audio Handle Declaration:
```c
// Global audio handle for SAI interface
AUDIO_HandleTypeDef haudio_sai;

// Global audio handle for I2S interface
AUDIO_HandleTypeDef haudio_i2s;
```

### Error Handling:
- All functions return `AUDIO_StatusTypeDef`
- Check return values: `AUDIO_OK`, `AUDIO_ERROR`, `AUDIO_BUSY`, `AUDIO_TIMEOUT`
- Use `AUDIO_GetStatusString()` for readable error messages

<a id='basic-playback'></a>
## 5. Basic Audio Playback

### Audio Configuration Structure:
```c
AUDIO_ConfigTypeDef audio_config = {
    .interface = AUDIO_INTERFACE_SAI,        // SAI or I2S
    .sample_rate = AUDIO_SAMPLE_RATE_44_1K,  // 44.1kHz
    .bit_depth = AUDIO_BIT_DEPTH_16,         // 16-bit
    .channels = AUDIO_CHANNELS_STEREO,       // Stereo
    .volume = 80,                            // 80% volume
    .dma_enabled = true                      // Use DMA
};
```

### Basic Playback Initialization:
```c
#include "audio.h"

int main(void) {
    // Configure audio for playback
    AUDIO_ConfigTypeDef config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_48K,
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_STEREO,
        .volume = 70,
        .dma_enabled = false  // Polling mode for simplicity
    };
    
    // Initialize audio system
    AUDIO_StatusTypeDef status = AUDIO_Init(&haudio_sai, &config);
    if (status != AUDIO_OK) {
        printf("Audio initialization failed: %s\n", 
               AUDIO_GetStatusString(status));
        return -1;
    }
    
    printf("Audio system initialized successfully\n");
    
    // Main application loop
    while (1) {
        // Application code here
    }
}
```

### Simple Tone Generation:
```c
// Generate a simple sine wave tone
#define SAMPLE_RATE 48000
#define FREQUENCY 440  // A4 note
#define AMPLITUDE 16384  // For 16-bit audio

void generate_tone(int16_t* buffer, uint32_t samples) {
    static float phase = 0.0f;
    float phase_increment = 2.0f * 3.14159f * FREQUENCY / SAMPLE_RATE;
    
    for (uint32_t i = 0; i < samples; i++) {
        buffer[i] = (int16_t)(AMPLITUDE * sinf(phase));
        phase += phase_increment;
        if (phase >= 2.0f * 3.14159f) {
            phase -= 2.0f * 3.14159f;
        }
    }
}

// Play the generated tone
int16_t audio_buffer[1024];  // 1KB buffer for stereo 16-bit

generate_tone(audio_buffer, 512);  // Generate 512 samples (mono)

AUDIO_StatusTypeDef status = AUDIO_PlayBuffer(&haudio_sai, 
                                             (uint8_t*)audio_buffer, 
                                             1024);  // 1024 bytes for stereo
```

### Playback Control:
```c
// Start playback
AUDIO_StartPlayback(&haudio_sai);

// Pause playback
AUDIO_PausePlayback(&haudio_sai);

// Resume playback
AUDIO_ResumePlayback(&haudio_sai);

// Stop playback
AUDIO_StopPlayback(&haudio_sai);
```

### Understanding Audio Data:
- **16-bit Stereo**: Each sample is 4 bytes (Left + Right channels)
- **Sample Rate**: 48kHz = 48,000 samples per second per channel
- **Data Rate**: 48kHz × 4 bytes = 192kB/s
- **Buffer Size**: Should be multiple of frame size (channels × bit_depth/8)

<a id='audio-recording'></a>
## 6. Audio Recording

### Recording Configuration:
```c
AUDIO_ConfigTypeDef record_config = {
    .interface = AUDIO_INTERFACE_SAI,
    .sample_rate = AUDIO_SAMPLE_RATE_16K,   // 16kHz for voice
    .bit_depth = AUDIO_BIT_DEPTH_16,
    .channels = AUDIO_CHANNELS_MONO,        // Mono for simplicity
    .volume = 100,                          // Maximum input gain
    .dma_enabled = true                     // DMA for continuous recording
};
```

### Basic Recording Setup:
```c
#include "audio.h"

#define RECORD_BUFFER_SIZE 4096
uint8_t record_buffer[RECORD_BUFFER_SIZE];

int main(void) {
    // Configure for recording
    AUDIO_ConfigTypeDef config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_16K,
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_MONO,
        .volume = 100,
        .dma_enabled = false
    };
    
    AUDIO_Init(&haudio_sai, &config);
    
    // Start recording
    AUDIO_StatusTypeDef status = AUDIO_StartRecording(&haudio_sai, 
                                                     record_buffer, 
                                                     RECORD_BUFFER_SIZE);
    if (status == AUDIO_OK) {
        printf("Recording started...\n");
        
        // Record for 2 seconds
        HAL_Delay(2000);
        
        // Stop recording
        AUDIO_StopRecording(&haudio_sai);
        printf("Recording stopped\n");
        
        // Process recorded data
        process_audio_data(record_buffer, RECORD_BUFFER_SIZE);
    }
}
```

### Recording with DMA (Continuous):
```c
// Circular buffer for continuous recording
#define CIRCULAR_BUFFER_SIZE 8192
uint8_t circular_buffer[CIRCULAR_BUFFER_SIZE];
volatile uint32_t record_index = 0;

void start_continuous_recording(void) {
    AUDIO_ConfigTypeDef config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_48K,
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_STEREO,
        .dma_enabled = true
    };
    
    AUDIO_Init(&haudio_sai, &config);
    AUDIO_StartRecording(&haudio_sai, circular_buffer, CIRCULAR_BUFFER_SIZE);
}

void process_recorded_data(void) {
    // Process latest recorded samples
    static uint32_t last_index = 0;
    
    while (last_index != record_index) {
        // Process audio sample
        int16_t sample = (int16_t)(circular_buffer[last_index] | 
                                  (circular_buffer[last_index + 1] << 8));
        
        // Apply processing (echo, filter, etc.)
        sample = process_sample(sample);
        
        last_index = (last_index + 2) % CIRCULAR_BUFFER_SIZE;  // 16-bit = 2 bytes
    }
}
```

### Audio Level Monitoring:
```c
// Monitor input audio levels
void monitor_audio_levels(uint8_t* buffer, uint32_t size) {
    int16_t max_level = 0;
    int16_t min_level = 32767;
    
    for (uint32_t i = 0; i < size; i += 2) {  // 16-bit samples
        int16_t sample = (int16_t)(buffer[i] | (buffer[i + 1] << 8));
        
        if (sample > max_level) max_level = sample;
        if (sample < min_level) min_level = sample;
    }
    
    // Convert to dBFS (decibels relative to full scale)
    float max_dB = 20.0f * log10f((float)abs(max_level) / 32767.0f);
    float min_dB = 20.0f * log10f((float)abs(min_level) / 32767.0f);
    
    printf("Audio Level: %.1f dBFS to %.1f dBFS\n", min_dB, max_dB);
}
```

### Recording Quality Considerations:
- **Sample Rate**: Higher rates capture more detail but use more bandwidth
- **Bit Depth**: Higher depth provides better dynamic range
- **Channels**: Mono uses half the bandwidth of stereo
- **Buffer Size**: Larger buffers prevent overflow but increase latency
- **Input Gain**: Proper gain prevents clipping and quantization noise

<a id='volume-control'></a>
## 7. Volume and Gain Control

### Volume Control Functions:
```c
// Set master volume (0-100%)
AUDIO_StatusTypeDef status = AUDIO_SetVolume(&haudio_sai, 75);
if (status != AUDIO_OK) {
    printf("Failed to set volume\n");
}

// Get current volume
uint8_t current_volume;
AUDIO_GetVolume(&haudio_sai, &current_volume);
printf("Current volume: %d%%\n", current_volume);
```

### Individual Channel Volume:
```c
// Set left channel volume
AUDIO_SetChannelVolume(&haudio_sai, AUDIO_CHANNEL_LEFT, 80);

// Set right channel volume
AUDIO_SetChannelVolume(&haudio_sai, AUDIO_CHANNEL_RIGHT, 60);

// Balance control
AUDIO_SetBalance(&haudio_sai, -20);  // -100 to +100 (left to right)
```

### Mute Control:
```c
// Mute audio output
AUDIO_Mute(&haudio_sai, true);

// Unmute audio output
AUDIO_Mute(&haudio_sai, false);

// Check mute status
bool is_muted;
AUDIO_IsMuted(&haudio_sai, &is_muted);
```

### Input Gain Control (Recording):
```c
// Set microphone gain
AUDIO_SetInputGain(&haudio_sai, AUDIO_INPUT_MIC, 50);  // 0-100%

// Set line input gain
AUDIO_SetInputGain(&haudio_sai, AUDIO_INPUT_LINE, 75);

// Automatic gain control
AUDIO_EnableAGC(&haudio_sai, true);
AUDIO_SetAGCLevel(&haudio_sai, 20);  // Target level in dB
```

### Volume Fading:
```c
// Fade volume over time
void fade_volume(uint8_t start_vol, uint8_t end_vol, uint32_t duration_ms) {
    uint32_t steps = 20;  // Number of fade steps
    uint32_t step_delay = duration_ms / steps;
    float vol_step = (float)(end_vol - start_vol) / steps;
    
    for (uint32_t i = 0; i <= steps; i++) {
        uint8_t current_vol = start_vol + (uint8_t)(vol_step * i);
        AUDIO_SetVolume(&haudio_sai, current_vol);
        HAL_Delay(step_delay);
    }
}

// Example: Fade in from silence to 80%
fade_volume(0, 80, 2000);  // 2-second fade in
```

### Digital Volume vs Analog Volume:
- **Digital Volume**: Applied in software, affects bit depth
- **Analog Volume**: Applied in codec hardware, preserves bit depth
- **Best Practice**: Use analog volume for main level, digital for fine tuning

### Volume Units:
- **Linear Scale**: 0-100% (what users expect)
- **dB Scale**: -60dB to +12dB (professional audio)
- **Codec Scale**: 0-255 (hardware specific)

### Volume Conversion:
```c
// Convert percentage to dB
float percent_to_db(uint8_t percent) {
    if (percent == 0) return -60.0f;  // Mute
    return 20.0f * log10f((float)percent / 100.0f);
}

// Convert dB to percentage
uint8_t db_to_percent(float db) {
    if (db <= -60.0f) return 0;
    return (uint8_t)(100.0f * powf(10.0f, db / 20.0f));
}
```

<a id='multi-channel'></a>
## 8. Multi-Channel Audio

### Multi-Channel Configuration:
```c
AUDIO_ConfigTypeDef multichannel_config = {
    .interface = AUDIO_INTERFACE_SAI,       // SAI supports TDM
    .sample_rate = AUDIO_SAMPLE_RATE_48K,
    .bit_depth = AUDIO_BIT_DEPTH_24,        // Higher quality
    .channels = AUDIO_CHANNELS_8,           // 8-channel TDM
    .volume = 80,
    .dma_enabled = true
};
```

### TDM (Time Division Multiplexing):
```c
// Configure TDM slots for 8-channel audio
AUDIO_TDMConfigTypeDef tdm_config = {
    .num_slots = 8,                         // 8 time slots
    .slot_size = AUDIO_SLOT_SIZE_32BIT,     // 32-bit slots
    .data_size = AUDIO_DATA_SIZE_24BIT,     // 24-bit data
    .frame_sync_width = 32,                 // Frame sync width
    .frame_length = 256                     // Total frame length
};

AUDIO_ConfigureTDM(&haudio_sai, &tdm_config);
```

### Multi-Channel Playback:
```c
// 8-channel audio buffer (24-bit samples)
#define CHANNELS 8
#define SAMPLES_PER_CHANNEL 1024
int32_t multichannel_buffer[CHANNELS * SAMPLES_PER_CHANNEL];

// Fill each channel with different content
for (int ch = 0; ch < CHANNELS; ch++) {
    for (int sample = 0; sample < SAMPLES_PER_CHANNEL; sample++) {
        // Generate different tones for each channel
        float frequency = 220.0f * (ch + 1);  // 220Hz, 440Hz, 660Hz, etc.
        float phase = 2.0f * 3.14159f * frequency * sample / 48000.0f;
        multichannel_buffer[ch * SAMPLES_PER_CHANNEL + sample] = 
            (int32_t)(8388607.0f * sinf(phase));  // 24-bit range
    }
}

// Play multi-channel audio
AUDIO_PlayMultichannel(&haudio_sai, (uint8_t*)multichannel_buffer, 
                      CHANNELS * SAMPLES_PER_CHANNEL * 4);  // 4 bytes per 24-bit sample
```

### Channel Routing and Mixing:
```c
// Route input channels to output channels
AUDIO_ChannelRouteTypeDef routing = {
    .input_channel = AUDIO_CHANNEL_0,
    .output_channel = AUDIO_CHANNEL_LEFT,
    .gain = 1.0f,                          // Unity gain
    .enable = true
};

AUDIO_SetChannelRouting(&haudio_sai, &routing);

// Mix multiple inputs to single output
AUDIO_MixChannels(&haudio_sai, 
                 AUDIO_CHANNEL_LEFT,       // Output channel
                 (AUDIO_ChannelTypeDef[]){AUDIO_CHANNEL_0, AUDIO_CHANNEL_1}, // Input channels
                 (float[]){0.7f, 0.3f},    // Mix ratios
                 2);                       // Number of inputs
```

### Surround Sound Setup:
```c
// 5.1 surround sound configuration
AUDIO_SurroundConfigTypeDef surround = {
    .front_left = AUDIO_CHANNEL_0,
    .front_right = AUDIO_CHANNEL_1,
    .center = AUDIO_CHANNEL_2,
    .subwoofer = AUDIO_CHANNEL_3,
    .rear_left = AUDIO_CHANNEL_4,
    .rear_right = AUDIO_CHANNEL_5,
    .enable_lfe = true,                    // Low frequency effects
    .lfe_gain = 0.8f
};

AUDIO_ConfigureSurround(&haudio_sai, &surround);
```

### Multi-Channel Recording:
```c
// Record from multiple microphones
#define MIC_CHANNELS 4
uint8_t mic_buffer[MIC_CHANNELS * 4096];  // 4KB per channel

AUDIO_StartMultichannelRecording(&haudio_sai, mic_buffer, 
                                MIC_CHANNELS * 4096, MIC_CHANNELS);

// Process each channel separately
for (int ch = 0; ch < MIC_CHANNELS; ch++) {
    uint8_t* channel_data = &mic_buffer[ch * 4096];
    process_microphone_data(channel_data, 4096, ch);
}
```

### Channel Considerations:
- **TDM Mode**: Required for more than 2 channels
- **Frame Sync**: Critical for channel synchronization
- **Buffer Alignment**: Data must be properly aligned for each channel
- **Processing Load**: Multi-channel increases CPU requirements
- **Memory Usage**: Larger buffers needed for multiple channels

<a id='dma-streaming'></a>
## 9. DMA Audio Streaming

### DMA Configuration:
```c
AUDIO_ConfigTypeDef dma_config = {
    .interface = AUDIO_INTERFACE_SAI,
    .sample_rate = AUDIO_SAMPLE_RATE_48K,
    .bit_depth = AUDIO_BIT_DEPTH_16,
    .channels = AUDIO_CHANNELS_STEREO,
    .volume = 80,
    .dma_enabled = true,                   // Enable DMA
    .buffer_mode = AUDIO_BUFFER_CIRCULAR   // Circular buffer
};
```

### Circular Buffer Streaming:
```c
#define AUDIO_BUFFER_SIZE 4096
uint8_t audio_buffer[AUDIO_BUFFER_SIZE];
volatile uint32_t buffer_offset = 0;

// Initialize DMA streaming
void init_audio_streaming(void) {
    AUDIO_ConfigTypeDef config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_48K,
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_STEREO,
        .dma_enabled = true,
        .buffer_mode = AUDIO_BUFFER_CIRCULAR
    };
    
    AUDIO_Init(&haudio_sai, &config);
    
    // Register callback for buffer half complete
    AUDIO_RegisterCallback(&haudio_sai, AUDIO_EVENT_HALF_COMPLETE, 
                          audio_half_callback);
    
    // Register callback for buffer complete
    AUDIO_RegisterCallback(&haudio_sai, AUDIO_EVENT_COMPLETE, 
                          audio_complete_callback);
}

// DMA callbacks
void audio_half_callback(AUDIO_HandleTypeDef* haudio) {
    // Fill first half of buffer
    fill_audio_buffer(&audio_buffer[0], AUDIO_BUFFER_SIZE / 2);
}

void audio_complete_callback(AUDIO_HandleTypeDef* haudio) {
    // Fill second half of buffer
    fill_audio_buffer(&audio_buffer[AUDIO_BUFFER_SIZE / 2], AUDIO_BUFFER_SIZE / 2);
}
```

### Double Buffering for Seamless Playback:
```c
// Double buffer implementation
#define BUFFER_COUNT 2
#define BUFFER_SIZE 2048
uint8_t audio_buffers[BUFFER_COUNT][BUFFER_SIZE];
volatile uint8_t active_buffer = 0;

// Fill buffer with audio data
void fill_audio_buffer(uint8_t* buffer, uint32_t size) {
    // Generate or load audio data
    for (uint32_t i = 0; i < size; i += 4) {  // Stereo 16-bit = 4 bytes per sample
        // Left channel
        int16_t left = generate_sample();
        buffer[i] = left & 0xFF;
        buffer[i + 1] = (left >> 8) & 0xFF;
        
        // Right channel
        int16_t right = generate_sample();
        buffer[i + 2] = right & 0xFF;
        buffer[i + 3] = (right >> 8) & 0xFF;
    }
}

// Start double-buffered playback
void start_double_buffer_playback(void) {
    // Fill both buffers initially
    fill_audio_buffer(audio_buffers[0], BUFFER_SIZE);
    fill_audio_buffer(audio_buffers[1], BUFFER_SIZE);
    
    // Start playback with first buffer
    AUDIO_PlayBuffer(&haudio_sai, audio_buffers[0], BUFFER_SIZE);
    
    // Set up DMA for continuous streaming
    AUDIO_EnableDMA(&haudio_sai, audio_buffers[active_buffer], BUFFER_SIZE);
}

void buffer_complete_callback(AUDIO_HandleTypeDef* haudio) {
    // Switch to next buffer
    active_buffer = (active_buffer + 1) % BUFFER_COUNT;
    
    // Fill the buffer that just finished playing
    fill_audio_buffer(audio_buffers[active_buffer], BUFFER_SIZE);
    
    // Continue with next buffer
    AUDIO_PlayBuffer(&haudio_sai, audio_buffers[active_buffer], BUFFER_SIZE);
}
```

### DMA for Recording:
```c
// Continuous recording with DMA
#define RECORD_BUFFER_SIZE 8192
uint8_t record_buffer[RECORD_BUFFER_SIZE];

void start_dma_recording(void) {
    AUDIO_ConfigTypeDef config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_16K,
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_MONO,
        .dma_enabled = true,
        .buffer_mode = AUDIO_BUFFER_CIRCULAR
    };
    
    AUDIO_Init(&haudio_sai, &config);
    
    // Start DMA recording
    AUDIO_StartRecording(&haudio_sai, record_buffer, RECORD_BUFFER_SIZE);
}

void recording_callback(AUDIO_HandleTypeDef* haudio) {
    // Process recorded data in real-time
    static uint32_t process_offset = 0;
    
    // Process latest chunk
    process_audio_chunk(&record_buffer[process_offset], RECORD_BUFFER_SIZE / 2);
    
    // Update offset for circular buffer
    process_offset = (process_offset + RECORD_BUFFER_SIZE / 2) % RECORD_BUFFER_SIZE;
}
```

### DMA Advantages:
- **CPU Free**: DMA handles data transfer automatically
- **Low Latency**: Minimal delay between audio events
- **High Bandwidth**: Supports high sample rates and bit depths
- **Continuous Streaming**: Seamless audio playback/recording
- **Buffer Management**: Automatic buffer switching

### DMA Considerations:
- **Memory Usage**: Large buffers consume RAM
- **Setup Complexity**: More complex initialization
- **Interrupt Handling**: Proper callback management required
- **Synchronization**: Buffer switching must be timed correctly
- **Debugging**: DMA issues can be hard to diagnose

<a id='audio-processing'></a>
## 10. Audio Processing

### Basic Filtering:
```c
// Simple low-pass filter
typedef struct {
    float alpha;        // Filter coefficient (0.0 to 1.0)
    float prev_output;  // Previous output sample
} LowPassFilter;

void init_lowpass_filter(LowPassFilter* filter, float cutoff_freq, float sample_rate) {
    // Calculate filter coefficient
    float rc = 1.0f / (2.0f * 3.14159f * cutoff_freq);
    float dt = 1.0f / sample_rate;
    filter->alpha = dt / (rc + dt);
    filter->prev_output = 0.0f;
}

int16_t process_lowpass_filter(LowPassFilter* filter, int16_t input) {
    // Apply filter: output = alpha * input + (1-alpha) * prev_output
    float output = filter->alpha * input + (1.0f - filter->alpha) * filter->prev_output;
    filter->prev_output = output;
    return (int16_t)output;
}

// Usage
LowPassFilter lpf;
init_lowpass_filter(&lpf, 1000.0f, 48000.0f);  // 1kHz cutoff at 48kHz

int16_t filtered_sample = process_lowpass_filter(&lpf, raw_sample);
```

### Echo/Delay Effect:
```c
#define DELAY_BUFFER_SIZE 48000  // 1 second at 48kHz
typedef struct {
    int16_t buffer[DELAY_BUFFER_SIZE];
    uint32_t write_index;
    uint32_t read_index;
    float feedback;     // 0.0 to 1.0
    float wet_dry;      // 0.0 (dry) to 1.0 (wet)
} DelayEffect;

void init_delay_effect(DelayEffect* delay, uint32_t delay_samples, float feedback, float wet_dry) {
    memset(delay->buffer, 0, sizeof(delay->buffer));
    delay->write_index = 0;
    delay->read_index = delay_samples;
    delay->feedback = feedback;
    delay->wet_dry = wet_dry;
}

int16_t process_delay_effect(DelayEffect* delay, int16_t input) {
    // Read delayed sample
    int16_t delayed = delay->buffer[delay->read_index];
    
    // Calculate output: mix dry and wet signals
    int16_t output = (int16_t)((1.0f - delay->wet_dry) * input + delay->wet_dry * delayed);
    
    // Write input + feedback to delay buffer
    delay->buffer[delay->write_index] = input + (int16_t)(delay->feedback * delayed);
    
    // Update indices
    delay->write_index = (delay->write_index + 1) % DELAY_BUFFER_SIZE;
    delay->read_index = (delay->read_index + 1) % DELAY_BUFFER_SIZE;
    
    return output;
}
```

### Audio Mixing:
```c
// Mix multiple audio streams
int16_t mix_audio_samples(int16_t sample1, int16_t sample2, float ratio1, float ratio2) {
    // Apply mixing ratios
    float mixed = ratio1 * sample1 + ratio2 * sample2;
    
    // Prevent clipping
    if (mixed > 32767.0f) mixed = 32767.0f;
    if (mixed < -32768.0f) mixed = -32768.0f;
    
    return (int16_t)mixed;
}

// Mix multiple channels
void mix_multichannel_audio(int16_t* output, int16_t** inputs, float* gains, 
                           uint32_t num_channels, uint32_t num_samples) {
    for (uint32_t sample = 0; sample < num_samples; sample++) {
        float mixed = 0.0f;
        
        for (uint32_t ch = 0; ch < num_channels; ch++) {
            mixed += gains[ch] * inputs[ch][sample];
        }
        
        // Apply compression/limiting if needed
        mixed = compressor_process(mixed);
        
        // Store result
        output[sample] = (int16_t)mixed;
    }
}
```

### Dynamic Range Compression:
```c
typedef struct {
    float threshold;    // Threshold in dB
    float ratio;        // Compression ratio
    float attack;       // Attack time in seconds
    float release;      // Release time in seconds
    float envelope;     // Current envelope
} Compressor;

void init_compressor(Compressor* comp, float threshold_db, float ratio, 
                    float attack_ms, float release_ms, float sample_rate) {
    comp->threshold = threshold_db;
    comp->ratio = ratio;
    comp->attack = 1.0f - expf(-1.0f / (attack_ms * 0.001f * sample_rate));
    comp->release = 1.0f - expf(-1.0f / (release_ms * 0.001f * sample_rate));
    comp->envelope = 0.0f;
}

int16_t process_compressor(Compressor* comp, int16_t input) {
    // Convert to dB
    float input_db = 20.0f * log10f(fabsf(input) / 32767.0f);
    
    // Calculate gain reduction
    float over_threshold = input_db - comp->threshold;
    float gain_reduction = 0.0f;
    
    if (over_threshold > 0.0f) {
        gain_reduction = over_threshold * (1.0f - 1.0f/comp->ratio);
    }
    
    // Smooth envelope
    if (gain_reduction > comp->envelope) {
        comp->envelope += comp->attack * (gain_reduction - comp->envelope);
    } else {
        comp->envelope += comp->release * (gain_reduction - comp->envelope);
    }
    
    // Apply gain reduction
    float output_db = input_db - comp->envelope;
    float output_linear = powf(10.0f, output_db / 20.0f);
    
    return (int16_t)(output_linear * 32767.0f * (input >= 0 ? 1.0f : -1.0f));
}
```

### Real-time Processing Pipeline:
```c
// Audio processing chain
typedef struct {
    LowPassFilter input_filter;
    DelayEffect echo;
    Compressor limiter;
    float master_gain;
} AudioProcessor;

void init_audio_processor(AudioProcessor* proc, float sample_rate) {
    // Initialize each effect
    init_lowpass_filter(&proc->input_filter, 15000.0f, sample_rate);  // High-pass
    init_delay_effect(&proc->echo, (uint32_t)(sample_rate * 0.3f), 0.4f, 0.3f);  // 300ms delay
    init_compressor(&proc->limiter, -6.0f, 4.0f, 5.0f, 100.0f, sample_rate);  // Compressor
    proc->master_gain = 0.8f;
}

int16_t process_audio_sample(AudioProcessor* proc, int16_t input) {
    float sample = input;
    
    // Apply processing chain
    sample = process_lowpass_filter(&proc->input_filter, sample);
    sample = process_delay_effect(&proc->echo, sample);
    sample = process_compressor(&proc->limiter, sample);
    sample *= proc->master_gain;
    
    // Prevent clipping
    if (sample > 32767.0f) sample = 32767.0f;
    if (sample < -32768.0f) sample = -32768.0f;
    
    return (int16_t)sample;
}
```

### Processing Considerations:
- **Latency**: Each effect adds processing delay
- **CPU Usage**: Complex effects require more processing power
- **Memory**: Delay buffers consume RAM
- **Precision**: Use floating point for intermediate calculations
- **Clipping**: Always prevent digital clipping

<a id='codec-integration'></a>
## 11. Codec Integration

### WM8994 Codec Configuration:
```c
// WM8994 codec handle
WM8994_HandleTypeDef hcodec;

// Initialize codec
void init_wm8994_codec(void) {
    hcodec.DeviceAddr = WM8994_ADDR_0;
    hcodec.OutputDevice = WM8994_OUT_HEADPHONE;
    hcodec.InputDevice = WM8994_IN_MIC1;
    hcodec.Frequency = WM8994_FREQUENCY_48K;
    hcodec.Resolution = WM8994_RESOLUTION_16B;
    hcodec.Volume = 80;
    
    // Initialize codec hardware
    if (WM8994_Init(&hcodec, &hi2c1) != WM8994_OK) {
        printf("Codec initialization failed\n");
        return;
    }
    
    // Configure audio interface
    WM8994_SetAudioInterface(&hcodec, WM8994_INTERFACE_I2S);
    
    // Set volume levels
    WM8994_SetVolume(&hcodec, WM8994_OUT_HEADPHONE, 80);
    WM8994_SetVolume(&hcodec, WM8994_IN_MIC1, 60);
}
```

### Codec Control Functions:
```c
// Set codec output device
WM8994_SetOutputMode(&hcodec, WM8994_OUT_HEADPHONE);  // Headphones
WM8994_SetOutputMode(&hcodec, WM8994_OUT_SPEAKER);    // Speaker

// Set codec input device
WM8994_SetInputMode(&hcodec, WM8994_IN_MIC1);         // Microphone 1
WM8994_SetInputMode(&hcodec, WM8994_IN_LINE1);        // Line input

// Control codec power
WM8994_PowerUp(&hcodec);                              // Power on
WM8994_PowerDown(&hcodec);                            // Power off
```

### Audio Path Configuration:
```c
// Configure playback path: DAC -> Headphone amplifier
WM8994_Play(&hcodec, WM8994_OUT_HEADPHONE);

// Configure recording path: Microphone -> ADC
WM8994_Record(&hcodec, WM8994_IN_MIC1);

// Configure bypass path: Line in -> Line out
WM8994_SetPath(&hcodec, WM8994_PATH_BYPASS);
```

### Codec Register Access:
```c
// Read codec register
uint16_t reg_value;
WM8994_ReadRegister(&hcodec, WM8994_REG_LEFT_LINE_INPUT_1_2_VOLUME, &reg_value);

// Write codec register
WM8994_WriteRegister(&hcodec, WM8994_REG_LEFT_OUTPUT_VOLUME, 0x50);

// Configure equalizer
WM8994_EnableEQ(&hcodec, true);
WM8994_SetEQBand(&hcodec, WM8994_EQ_BAND_BASS, 12);    // +12dB bass boost
WM8994_SetEQBand(&hcodec, WM8994_EQ_BAND_TREBLE, -6);  // -6dB treble cut
```

### Codec Synchronization:
```c
// Ensure codec and SAI are synchronized
void synchronize_codec_and_sai(void) {
    // Configure SAI first
    AUDIO_ConfigTypeDef sai_config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_48K,
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_STEREO
    };
    AUDIO_Init(&haudio_sai, &sai_config);
    
    // Configure codec to match
    hcodec.Frequency = WM8994_FREQUENCY_48K;
    hcodec.Resolution = WM8994_RESOLUTION_16B;
    WM8994_Init(&hcodec, &hi2c1);
    
    // Start both simultaneously
    AUDIO_StartPlayback(&haudio_sai);
    WM8994_Play(&hcodec, WM8994_OUT_HEADPHONE);
}
```

### Multiple Codec Support:
```c
// Support for multiple codecs on same bus
#define CODEC_COUNT 2
WM8994_HandleTypeDef codecs[CODEC_COUNT];

void init_multiple_codecs(void) {
    // Codec 1 - Address 0x1A
    codecs[0].DeviceAddr = WM8994_ADDR_0;
    WM8994_Init(&codecs[0], &hi2c1);
    
    // Codec 2 - Address 0x1B (if available)
    codecs[1].DeviceAddr = WM8994_ADDR_1;
    WM8994_Init(&codecs[1], &hi2c1);
    
    // Configure different outputs
    WM8994_SetOutputMode(&codecs[0], WM8994_OUT_HEADPHONE);
    WM8994_SetOutputMode(&codecs[1], WM8994_OUT_SPEAKER);
}
```

### Codec Diagnostics:
```c
// Check codec status
void diagnose_codec(void) {
    uint16_t reg_val;
    
    // Check codec ID
    WM8994_ReadRegister(&hcodec, WM8994_REG_SOFTWARE_RESET, &reg_val);
    printf("Codec ID: 0x%04X\n", reg_val);
    
    // Check PLL lock status
    WM8994_ReadRegister(&hcodec, WM8994_REG_PLL1_K_3, &reg_val);
    if (reg_val & 0x8000) {
        printf("PLL locked\n");
    } else {
        printf("PLL not locked - check clock input\n");
    }
    
    // Check power status
    WM8994_ReadRegister(&hcodec, WM8994_REG_POWER_MANAGEMENT_1, &reg_val);
    printf("Power status: 0x%04X\n", reg_val);
}
```

### Codec Integration Best Practices:
- **Initialization Order**: Configure SAI first, then codec
- **Clock Synchronization**: Ensure MCLK is stable before codec init
- **Power Sequencing**: Follow codec power-up/power-down sequence
- **Register Caching**: Cache frequently read registers
- **Error Recovery**: Implement codec reset on communication errors
- **Temperature Compensation**: Monitor and compensate for temperature effects

<a id='advanced'></a>
## 12. Advanced Features

### Interrupt-Driven Audio:
```c
// Register audio event callbacks
void audio_event_callback(AUDIO_HandleTypeDef* haudio, AUDIO_EventTypeDef event) {
    switch (event) {
        case AUDIO_EVENT_HALF_COMPLETE:
            // Fill first half of DMA buffer
            fill_audio_buffer(&audio_buffer[0], BUFFER_SIZE / 2);
            break;
            
        case AUDIO_EVENT_COMPLETE:
            // Fill second half of DMA buffer
            fill_audio_buffer(&audio_buffer[BUFFER_SIZE / 2], BUFFER_SIZE / 2);
            break;
            
        case AUDIO_EVENT_ERROR:
            // Handle audio error
            AUDIO_ErrorHandler(haudio);
            break;
            
        case AUDIO_EVENT_OVERFLOW:
            // Handle buffer overflow
            AUDIO_StopPlayback(haudio);
            AUDIO_StartPlayback(haudio);  // Restart
            break;
    }
}

// Register callbacks
AUDIO_RegisterCallback(&haudio_sai, AUDIO_EVENT_ALL, audio_event_callback);
```

### Audio Statistics and Monitoring:
```c
// Get audio performance statistics
AUDIO_StatsTypeDef stats;
AUDIO_GetStatistics(&haudio_sai, &stats);

printf("Audio Statistics:\n");
printf("Total samples played: %lu\n", stats.samples_played);
printf("Buffer underruns: %lu\n", stats.underruns);
printf("Buffer overruns: %lu\n", stats.overruns);
printf("Average CPU usage: %.1f%%\n", stats.cpu_usage);

// Monitor audio levels
AUDIO_LevelTypeDef levels;
AUDIO_GetLevels(&haudio_sai, &levels);

printf("Audio Levels:\n");
printf("Left peak: %.1f dBFS\n", levels.left_peak);
printf("Right peak: %.1f dBFS\n", levels.right_peak);
printf("RMS level: %.1f dBFS\n", levels.rms_level);
```

### Audio Format Conversion:
```c
// Convert between different audio formats
void convert_audio_format(uint8_t* input, uint8_t* output, 
                         AUDIO_FormatTypeDef input_format, 
                         AUDIO_FormatTypeDef output_format,
                         uint32_t num_samples) {
    
    for (uint32_t i = 0; i < num_samples; i++) {
        // Read input sample
        int32_t sample = 0;
        
        switch (input_format) {
            case AUDIO_FORMAT_16BIT:
                sample = (int16_t)(input[i*2] | (input[i*2+1] << 8));
                break;
            case AUDIO_FORMAT_24BIT:
                sample = (int32_t)(input[i*3] | (input[i*3+1] << 8) | (input[i*3+2] << 16));
                if (sample & 0x800000) sample |= 0xFF000000;  // Sign extend
                break;
            case AUDIO_FORMAT_32BIT:
                sample = (int32_t)(input[i*4] | (input[i*4+1] << 8) | 
                                   (input[i*4+2] << 16) | (input[i*4+3] << 24));
                break;
        }
        
        // Write output sample
        switch (output_format) {
            case AUDIO_FORMAT_16BIT:
                output[i*2] = sample & 0xFF;
                output[i*2+1] = (sample >> 8) & 0xFF;
                break;
            case AUDIO_FORMAT_24BIT:
                output[i*3] = sample & 0xFF;
                output[i*3+1] = (sample >> 8) & 0xFF;
                output[i*3+2] = (sample >> 16) & 0xFF;
                break;
            case AUDIO_FORMAT_32BIT:
                output[i*4] = sample & 0xFF;
                output[i*4+1] = (sample >> 8) & 0xFF;
                output[i*4+2] = (sample >> 16) & 0xFF;
                output[i*4+3] = (sample >> 24) & 0xFF;
                break;
        }
    }
}
```

### Audio Streaming over Network:
```c
// Stream audio over Ethernet/UDP
#define AUDIO_STREAM_PORT 50000
#define PACKET_SIZE 1024

void init_audio_streaming(void) {
    // Initialize network interface
    // Initialize UDP socket
    
    // Configure audio for streaming
    AUDIO_ConfigTypeDef config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_16K,  // Lower rate for network
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_MONO,
        .dma_enabled = true
    };
    
    AUDIO_Init(&haudio_sai, &config);
    AUDIO_StartRecording(&haudio_sai, stream_buffer, PACKET_SIZE);
}

void stream_audio_packet(uint8_t* data, uint32_t size) {
    // Compress audio if needed
    // Encrypt if required
    // Send over network
    
    UDP_SendPacket(audio_socket, data, size);
}
```

### Power Management:
```c
// Audio power management
void enter_audio_standby(void) {
    // Mute outputs
    AUDIO_Mute(&haudio_sai, true);
    
    // Stop audio processing
    AUDIO_StopPlayback(&haudio_sai);
    AUDIO_StopRecording(&haudio_sai);
    
    // Power down codec
    WM8994_PowerDown(&hcodec);
    
    // Disable SAI peripheral
    AUDIO_DeInit(&haudio_sai);
}

void exit_audio_standby(void) {
    // Reinitialize audio system
    AUDIO_Init(&haudio_sai, &audio_config);
    WM8994_Init(&hcodec, &hi2c1);
    
    // Fade in volume
    AUDIO_Mute(&haudio_sai, false);
    fade_volume(0, 80, 500);
}
```

### Audio Testing and Calibration:
```c
// Generate test signals
void generate_test_signal(AUDIO_TestSignalTypeDef signal_type, 
                         uint8_t* buffer, uint32_t size) {
    
    for (uint32_t i = 0; i < size; i += 2) {
        int16_t sample = 0;
        
        switch (signal_type) {
            case AUDIO_TEST_SINE:
                sample = (int16_t)(16384 * sinf(2 * 3.14159f * 1000 * i / 48000));
                break;
            case AUDIO_TEST_SQUARE:
                sample = (i % (48000 / 1000) < 24000 / 1000) ? 16384 : -16384;
                break;
            case AUDIO_TEST_NOISE:
                sample = rand() % 32768 - 16384;
                break;
            case AUDIO_TEST_SWEEP:
                // Frequency sweep implementation
                break;
        }
        
        buffer[i] = sample & 0xFF;
        buffer[i+1] = (sample >> 8) & 0xFF;
    }
}

// Audio calibration
void calibrate_audio_system(void) {
    // Generate 1kHz test tone at -20dBFS
    generate_test_signal(AUDIO_TEST_SINE, cal_buffer, CAL_BUFFER_SIZE);
    
    // Play calibration tone
    AUDIO_PlayBuffer(&haudio_sai, cal_buffer, CAL_BUFFER_SIZE);
    
    // Measure output level with external equipment
    // Adjust codec gain registers accordingly
    
    printf("Adjust output level to -20dBFS and press enter\n");
    getchar();
    
    // Store calibration values in non-volatile memory
    save_calibration_data();
}
```

### Advanced Audio Processing:
```c
// Real-time FFT analysis
#define FFT_SIZE 1024
float fft_input[FFT_SIZE];
float fft_output[FFT_SIZE];

void perform_fft_analysis(int16_t* audio_samples, uint32_t num_samples) {
    // Convert to float and apply window
    for (uint32_t i = 0; i < num_samples; i++) {
        fft_input[i] = (float)audio_samples[i] / 32768.0f;
        fft_input[i] *= 0.5f * (1.0f - cosf(2.0f * 3.14159f * i / (num_samples - 1)));  // Hann window
    }
    
    // Perform FFT (using CMSIS-DSP library)
    arm_rfft_fast_f32(&fft_instance, fft_input, fft_output, 0);
    
    // Calculate magnitude spectrum
    for (uint32_t i = 0; i < num_samples / 2; i++) {
        float real = fft_output[2 * i];
        float imag = fft_output[2 * i + 1];
        float magnitude = sqrtf(real * real + imag * imag);
        
        // Use magnitude for spectrum analysis, equalization, etc.
        process_frequency_bin(i, magnitude);
    }
}
```

### Performance Optimization:
- **Use DMA** for all high-bandwidth audio operations
- **Optimize buffer sizes** for specific use cases
- **Implement double buffering** for seamless streaming
- **Use fixed-point arithmetic** where possible
- **Profile and optimize** critical audio processing loops
- **Consider cache alignment** for audio buffers

<a id='troubleshooting'></a>
## 13. Troubleshooting

### Common Issues and Solutions:

#### 1. No Audio Output
**Symptoms:** Audio plays but no sound from speakers/headphones
**Possible Causes:**
- Codec not initialized or configured incorrectly
- Wrong output device selected (headphone vs speaker)
- Muted outputs or zero volume
- SAI pins not configured correctly
- Codec power issues
**Solutions:**
- Check codec initialization status
- Verify output device selection with `WM8994_SetOutputMode()`
- Check volume levels with `AUDIO_GetVolume()`
- Verify SAI pin configuration in STM32CubeMX
- Check codec power supply and I2C communication

#### 2. Distorted Audio
**Symptoms:** Audio sounds distorted or crackly
**Possible Causes:**
- Buffer underruns (DMA too slow)
- Incorrect sample rate/bit depth mismatch
- Codec configuration errors
- Power supply noise
- Ground loops
**Solutions:**
- Increase buffer sizes for DMA
- Verify sample rate/bit depth consistency between SAI and codec
- Check codec register settings
- Add decoupling capacitors
- Improve grounding and shielding

#### 3. Recording Issues
**Symptoms:** Cannot record audio or poor quality recordings
**Possible Causes:**
- Wrong input device selected
- Microphone gain too low/high
- Incorrect SAI configuration for input
- Codec ADC not enabled
- Input signal too weak
**Solutions:**
- Check input device selection with `WM8994_SetInputMode()`
- Adjust microphone gain with `AUDIO_SetInputGain()`
- Verify SAI configuration for recording
- Enable codec ADC channels
- Check input signal levels

#### 4. Synchronization Problems
**Symptoms:** Audio glitches or timing issues
**Possible Causes:**
- SAI and codec clock mismatch
- DMA buffer management issues
- Interrupt timing problems
- PLL not locked
**Solutions:**
- Ensure SAI and codec use same sample rate
- Check DMA buffer sizes and alignment
- Verify interrupt priorities
- Check PLL status and clock inputs

#### 5. High CPU Usage
**Symptoms:** System becomes unresponsive during audio processing
**Possible Causes:**
- Polling mode instead of DMA
- Complex audio processing algorithms
- Small buffer sizes causing frequent interrupts
- Inefficient code in audio callbacks
**Solutions:**
- Enable DMA for all audio operations
- Optimize audio processing algorithms
- Increase buffer sizes to reduce interrupt frequency
- Profile and optimize callback functions

#### 6. I2C Communication Errors
**Symptoms:** Codec initialization fails
**Possible Causes:**
- Wrong I2C address
- I2C bus issues (pull-ups, speed)
- Codec not powered or reset
- Pin conflicts
**Solutions:**
- Verify codec I2C address (0x1A or 0x1B)
- Check I2C pull-up resistors and bus speed
- Ensure proper power sequencing
- Check for pin conflicts in STM32CubeMX

#### 7. DMA Transfer Errors
**Symptoms:** Audio stuttering or dropouts
**Possible Causes:**
- DMA channel conflicts
- Incorrect DMA configuration
- Memory alignment issues
- Buffer size mismatches
**Solutions:**
- Check DMA channel allocation
- Verify DMA stream and channel configuration
- Ensure buffers are properly aligned
- Match buffer sizes between DMA and audio

### Debug Tips:
```c
// Check audio system status
AUDIO_StatusTypeDef status = AUDIO_GetStatus(&haudio_sai);
printf("Audio Status: %s\n", AUDIO_GetStatusString(status));

// Verify codec communication
uint16_t codec_id;
if (WM8994_ReadRegister(&hcodec, WM8994_REG_SOFTWARE_RESET, &codec_id) == WM8994_OK) {
    printf("Codec ID: 0x%04X (expected: 0x8994)\n", codec_id);
} else {
    printf("Codec communication failed\n");
}

// Check SAI registers
printf("SAI GCR: 0x%08X\n", SAI1->GCR);
printf("SAI CR1: 0x%08X\n", SAI1_Block_A->CR1);
printf("SAI SR: 0x%08X\n", SAI1_Block_A->SR);

// Monitor audio buffer status
AUDIO_BufferStatusTypeDef buf_status;
AUDIO_GetBufferStatus(&haudio_sai, &buf_status);
printf("Buffer usage: %d%%\n", buf_status.usage_percent);

// Test basic audio functionality
uint8_t test_buffer[1024];
memset(test_buffer, 0x80, sizeof(test_buffer));  // Middle value
if (AUDIO_PlayBuffer(&haudio_sai, test_buffer, sizeof(test_buffer)) == AUDIO_OK) {
    printf("Basic playback test passed\n");
} else {
    printf("Basic playback test failed\n");
}
```

### Performance Optimization Tips:
- **Use DMA** for all audio data transfer
- **Optimize buffer sizes** based on latency requirements
- **Implement circular buffers** for continuous streaming
- **Minimize interrupt service routines**
- **Use appropriate sample rates** for the application
- **Profile audio processing** to identify bottlenecks
- **Consider fixed-point math** for DSP operations

### Hardware Debugging:
- **Oscilloscope**: Check SAI signals (MCK, SCK, FS, SD)
- **Logic Analyzer**: Verify I2S/SAI protocol compliance
- **Audio Analyzer**: Measure frequency response and distortion
- **Power Supply**: Monitor voltage levels and ripple
- **Spectrum Analyzer**: Check for noise and interference

### Getting Help:
- Check STM32F429 Reference Manual (SAI chapter)
- Review codec datasheet (WM8994 or equivalent)
- Verify HAL driver implementation
- Test with known working hardware
- Check audio forum communities for similar issues

For more detailed information, refer to the STM32F429 datasheet, SAI/I2S specifications, and codec reference manuals.

## Complete Example Program

```c
#include "audio.h"
#include "wm8994.h"
#include "stm32f4xx.h"
#include <stdio.h>
#include <math.h>

// Audio configuration
#define AUDIO_SAMPLE_RATE    48000
#define AUDIO_BUFFER_SIZE    4096
#define AUDIO_CHANNELS       2

// Global handles
AUDIO_HandleTypeDef haudio;
WM8994_HandleTypeDef hcodec;
I2C_HandleTypeDef hi2c1;

// Audio buffers
uint8_t audio_buffer[AUDIO_BUFFER_SIZE];
volatile uint32_t buffer_offset = 0;

// Sine wave generation for testing
float phase = 0.0f;
const float frequency = 440.0f;  // A4 note

void generate_sine_wave(uint8_t* buffer, uint32_t size) {
    for (uint32_t i = 0; i < size; i += 4) {  // Stereo 16-bit = 4 bytes
        // Generate sine wave sample
        int16_t sample = (int16_t)(16384 * sinf(phase));
        phase += 2.0f * 3.14159f * frequency / AUDIO_SAMPLE_RATE;
        
        if (phase >= 2.0f * 3.14159f) {
            phase -= 2.0f * 3.14159f;
        }
        
        // Left channel
        buffer[i] = sample & 0xFF;
        buffer[i + 1] = (sample >> 8) & 0xFF;
        
        // Right channel (same as left for mono sound)
        buffer[i + 2] = sample & 0xFF;
        buffer[i + 3] = (sample >> 8) & 0xFF;
    }
}

void audio_half_callback(AUDIO_HandleTypeDef* haudio) {
    // Fill first half of buffer
    generate_sine_wave(&audio_buffer[0], AUDIO_BUFFER_SIZE / 2);
}

void audio_complete_callback(AUDIO_HandleTypeDef* haudio) {
    // Fill second half of buffer
    generate_sine_wave(&audio_buffer[AUDIO_BUFFER_SIZE / 2], AUDIO_BUFFER_SIZE / 2);
}

int main(void) {
    printf("STM32F429 Audio Demonstration\n\n");
    
    // Initialize I2C for codec communication
    hi2c1.Instance = I2C1;
    hi2c1.Init.ClockSpeed = 100000;
    hi2c1.Init.DutyCycle = I2C_DUTYCYCLE_2;
    hi2c1.Init.OwnAddress1 = 0;
    hi2c1.Init.AddressingMode = I2C_ADDRESSINGMODE_7BIT;
    hi2c1.Init.DualAddressMode = I2C_DUALADDRESS_DISABLE;
    hi2c1.Init.OwnAddress2 = 0;
    hi2c1.Init.GeneralCallMode = I2C_GENERALCALL_DISABLE;
    hi2c1.Init.NoStretchMode = I2C_NOSTRETCH_DISABLE;
    
    if (HAL_I2C_Init(&hi2c1) != HAL_OK) {
        printf("I2C initialization failed\n");
        return -1;
    }
    
    // Initialize codec
    hcodec.DeviceAddr = WM8994_ADDR_0;
    hcodec.OutputDevice = WM8994_OUT_HEADPHONE;
    hcodec.InputDevice = WM8994_IN_NONE;
    hcodec.Frequency = WM8994_FREQUENCY_48K;
    hcodec.Resolution = WM8994_RESOLUTION_16B;
    hcodec.Volume = 80;
    
    if (WM8994_Init(&hcodec, &hi2c1) != WM8994_OK) {
        printf("Codec initialization failed\n");
        return -1;
    }
    printf("Codec initialized successfully\n");
    
    // Configure audio system
    AUDIO_ConfigTypeDef audio_config = {
        .interface = AUDIO_INTERFACE_SAI,
        .sample_rate = AUDIO_SAMPLE_RATE_48K,
        .bit_depth = AUDIO_BIT_DEPTH_16,
        .channels = AUDIO_CHANNELS_STEREO,
        .volume = 70,
        .dma_enabled = true,
        .buffer_mode = AUDIO_BUFFER_CIRCULAR
    };
    
    AUDIO_StatusTypeDef status = AUDIO_Init(&haudio, &audio_config);
    if (status != AUDIO_OK) {
        printf("Audio initialization failed: %s\n", AUDIO_GetStatusString(status));
        return -1;
    }
    printf("Audio system initialized successfully\n\n");
    
    // Register audio callbacks
    AUDIO_RegisterCallback(&haudio, AUDIO_EVENT_HALF_COMPLETE, audio_half_callback);
    AUDIO_RegisterCallback(&haudio, AUDIO_EVENT_COMPLETE, audio_complete_callback);
    
    // Fill initial buffer
    generate_sine_wave(audio_buffer, AUDIO_BUFFER_SIZE);
    
    // Start audio playback
    status = AUDIO_StartPlayback(&haudio);
    if (status != AUDIO_OK) {
        printf("Failed to start playback: %s\n", AUDIO_GetStatusString(status));
        return -1;
    }
    printf("Audio playback started - you should hear a 440Hz tone\n");
    printf("Press any key to stop...\n");
    
    // Wait for user input
    getchar();
    
    // Stop playback
    AUDIO_StopPlayback(&haudio);
    printf("Audio playback stopped\n");
    
    // Demonstrate recording
    printf("\nDemonstrating audio recording...\n");
    
    uint8_t record_buffer[2048];
    status = AUDIO_StartRecording(&haudio, record_buffer, sizeof(record_buffer));
    if (status == AUDIO_OK) {
        printf("Recording for 2 seconds...\n");
        HAL_Delay(2000);
        AUDIO_StopRecording(&haudio);
        printf("Recording complete\n");
        
        // Process recorded data (simple level detection)
        int16_t max_level = 0;
        for (uint32_t i = 0; i < sizeof(record_buffer); i += 2) {
            int16_t sample = (int16_t)(record_buffer[i] | (record_buffer[i + 1] << 8));
            if (abs(sample) > max_level) max_level = abs(sample);
        }
        
        float level_db = 20.0f * log10f((float)max_level / 32767.0f);
        printf("Recorded audio peak level: %.1f dBFS\n", level_db);
    }
    
    printf("\nAudio demonstration complete!\n");
    
    return 0;
}
```

## Summary

This tutorial covered:
- Audio system fundamentals and STM32F429 audio capabilities
- Hardware setup with SAI/I2S interfaces and codec integration
- Basic audio playback and recording operations
- Volume control and gain management
- Multi-channel audio and surround sound
- DMA streaming for high-performance audio
- Audio processing (filtering, effects, mixing)
- Codec integration with WM8994
- Advanced features (interrupts, statistics, format conversion)
- Troubleshooting common audio issues

### Key Takeaways:
1. **Always synchronize** SAI and codec configurations
2. **Use DMA** for continuous audio streaming
3. **Check return values** from all audio functions
4. **Implement proper error handling** for robust operation
5. **Consider buffer sizes** for latency vs memory trade-offs
6. **Test incrementally** when setting up audio systems

### Next Steps:
- Experiment with different sample rates and bit depths
- Implement audio effects and processing algorithms
- Add network streaming capabilities
- Integrate with audio analysis libraries
- Develop custom audio applications

### Additional Resources:
- STM32F429 Reference Manual (SAI chapter)
- WM8994 Codec Datasheet
- I2S and SAI protocol specifications
- Audio processing algorithm references
- Digital signal processing tutorials

For more detailed information, refer to the STM32F429 datasheet, audio codec datasheets, and digital audio standards documentation.